# Sesión práctica 2 · Búsqueda semántica a escala con FAISS

## Del embedding al producto que termina apareciendo en pantalla

Un encoder transforma una consulta y cada producto en vectores. Eso no resuelve por sí solo la búsqueda. Todavía hace falta localizar los $k$ productos más próximos, conservar la relación entre posiciones vectoriales e identificadores de negocio, recuperar los metadatos, aplicar restricciones y ordenar la respuesta final.

En un catálogo pequeño se puede comparar la consulta con todos los vectores. Esa búsqueda exacta es sencilla y produce el vecino real según la métrica elegida. Su coste, sin embargo, crece con el número de productos $N$, la dimensión $d$ y el número de consultas $Q$. Para un lote de consultas, la operación dominante puede expresarse como una multiplicación $QX^\top$ de coste aproximado $O(QNd)$.

Los algoritmos **Approximate Nearest Neighbor (ANN)** reducen el trabajo evitando comparar contra todo el catálogo. A cambio, pueden omitir vecinos que el cálculo exacto habría devuelto. La palabra *aproximado* no se refiere al embedding ni a que el score tenga pocos decimales: se refiere a que el algoritmo de recuperación renuncia a la garantía de encontrar siempre el top-$k$ exacto.

El marketplace utilizado en nuestro ejemplo contiene 50.000 productos españoles reales del dataset **Shopping Queries Dataset** de Amazon Science. Las consultas incluyen intenciones literales, paráfrasis por necesidad y 256 ejemplos de producto utilizados como test. El objetivo de esta sesión será medir tres recursos que entran en conflicto conforme nuestro marketplace escala: fidelidad del ranking, latencia y memoria.


## Índice de contenidos

1. [Del embedding al sistema de recuperación](#1-del-embedding-al-sistema-de-recuperación)
2. [Búsqueda exacta](#2-búsqueda-exacta)
3. [Qué significa buscar aproximadamente](#3-qué-significa-buscar-aproximadamente)
4. [IVF](#4-ivf-buscar-primero-la-región-del-espacio)
5. [HNSW](#5-hnsw-navegar-por-un-grafo-de-proximidad)
6. [Product Quantization](#6-product-quantization-comprimir-para-poder-buscar)
7. [Comparación de configuraciones](#7-comparar-configuraciones-la-frontera-de-pareto)
8. [Metadatos, filtros y persistencia](#8-metadatos-filtros-y-persistencia)
9. [Decisión para el marketplace](#9-decisión-para-el-marketplace)


## 1. Del embedding al sistema de recuperación

### 1.1. Dos evaluaciones distintas que no deben mezclarse

La calidad de un buscador semántico tiene al menos dos capas:

* **Calidad del modelo.** Un embedding puede colocar juntos productos irrelevantes. Esa capa se evalúa con juicios de negocio (como `Exact`, `Substitute`, `Complement` e `Irrelevant` en nuestro ejemplo) y métricas como nDCG. Un índice exacto no arregla una mala geometría: encontrará con precisión los vecinos equivocados del modelo.

* **Fidelidad del índice.** Dado un espacio vectorial fijo, un ANN puede no devolver algunos vecinos que devolvería la fuerza bruta. Esa capa se evalúa comparando el top-$k$ aproximado contra el top-$k$ exacto mediante recall@k.
  $$
  \operatorname{recall@k}(q)=
  \frac{|ANN_k(q)\cap Exact_k(q)|}{k}
  $$
  Un recall@10 de 0,9 significa que, en promedio, nueve de los diez IDs exactos aparecen en el top diez aproximado. No significa que el buscador tenga un 90 % de relevancia ni que el 90 % de los usuarios compre. El recall del índice utiliza el ranking exacto como oráculo técnico; nDCG utiliza juicios humanos como referencia de negocio.

Esta separación permite atribuir los errores. Si un índice con búsqueda exacta como `IndexFlatIP` ya devuelve malos productos, el problema está en la representación: lo ocasiona el modelo de embeddings o la forma de tratar los datos. Por otro lado, si Flat funciona y un algoritmo ANN como podría ser IVF con `nprobe=1` pierde resultados, el problema se ha introducido en la aproximación.

Para ver estos conceptos, empezaremos cargando el entorno, los 50.000 productos, las consultas y las matrices de embeddings. Antes de construir un índice comprobaremos formas, tipos y metadatos; la búsqueda solo será interpretable si esos artefactos comparten el mismo contrato.


In [1]:
from pathlib import Path
import os
import sys

current_directory = Path.cwd().resolve()
project_root = next(
    candidate
    for candidate in (current_directory, *current_directory.parents)
    if (candidate / "pyproject.toml").exists()
)
sys.path.insert(0, str(project_root / "src"))


In [2]:
from time import perf_counter

import faiss
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from dotenv import load_dotenv

load_dotenv(project_root / ".env", override=False)
pio.templates.default = "plotly_white"


In [ ]:
from vector_index_session import (
    benchmark_search,
    build_flat_index,
    build_hnsw_index,
    build_ivf_flat_index,
    build_ivf_pq_index,
    configure_faiss_threads,
    load_session_data,
    recall_per_query,
    serialized_size_bytes,
)

# faiss_threads = configure_faiss_threads()
session_data = load_session_data(memory_map=True)


In [4]:
products = session_data.products
queries = session_data.queries
judgments = session_data.judgments
product_embeddings = session_data.product_embeddings
query_embeddings = session_data.query_embeddings

print(f"Productos: {len(products):,}")
print(f"Consultas de benchmark: {len(queries):,}")
print(f"Dimensión de los embeddings: {product_embeddings.shape[1]:,}")
# print(f"Threads FAISS: {faiss_threads}")


Productos: 50,000
Consultas de benchmark: 276
Dimensión de los embeddings: 384


### 1.2. El flujo completo de una consulta

El índice vectorial ocupa una posición concreta dentro del sistema. No tokeniza texto, no llama al encoder y no conoce precios, marcas ni stock salvo que otra capa se los proporcione. FAISS recibe matrices numéricas `float32` y devuelve scores e IDs enteros.

Durante la indexación se prepara el texto del producto, se calcula su embedding, se normaliza si la métrica lo requiere y se añade al índice. En paralelo se guarda una tabla que traduce el ID entero interno a `product_id` y metadatos.

Durante la consulta se reproduce exactamente el contrato del encoder, se busca el top-$k$, se complementan los IDs con la tabla de productos y se aplican reglas posteriores: disponibilidad, permisos, filtros, diversidad o reranking. Si se cambia el modelo, la dimensión, la normalización o la plantilla de entrada, el índice deja de ser compatible aunque la API de FAISS siga aceptando el array.


### 1.3. El ciclo de vida `train → add → search`

Todos los índices de FAISS comparten una interfaz, pero no todos atraviesan las mismas fases. Por ejemplo, `IndexFlatIP` nace entrenado porque no tiene parámetros estadísticos que aprender. Se crea con la dimensión, se añaden vectores y puede buscar inmediatamente.

Por otro lado, índices como **Inverted File Index (IVF)** o **Product Quantization (PQ)** necesitan una fase de entrenamiento. Durante esta fase aprenden los parámetros o características necesarias utilizando una muestra representativa de los datos. Una vez finalizado el entrenamiento, el atributo `is_trained` cambia a `True`, pero el índice todavía no contiene productos: el atributo `ntotal` sigue en cero. El método `add` asigna y codifica cada vector usando los parámetros ya aprendidos. Es importante no confundir la fase de entrenamiento con la fase de indexación en este tipo de índices.

Existen otros índices, como **Hierarchical Navigable Small World (HNSW)**, donde no se ejecuta un entrenamiento estadístico separado. El trabajo en HNSW ocurre durante la fase de indexación (al usar el método `add`), cuando cada nuevo vector navega el grafo existente, selecciona vecinos y crea enlaces.

Cuando ya tenemos el índice entrenado e indexado, el método `search` nos permite realizar el proceso de búsqueda. Este exige consultas `float32` contiguas con la misma dimensión. 

> Nótese que FAISS no comprueba que procedan del mismo encoder ni que utilicen la misma normalización. Dos matrices de 384 columnas pueden ser incompatibles semánticamente y, aun así, producir scores sin ningún error de ejecución.

Representaremos este ciclo de vida como una pipeline completa, desde el texto de la query hasta la complementación de productos. El diagrama nos servirá para ubicar después qué coste y qué posible fallo pertenece a cada etapa.


```mermaid
flowchart TB
    subgraph BUILD["Construcción o actualización del índice"]
        direction LR

        CATALOG["Embeddings del catálogo<br/>float32 · dimensión d"]
        INDEX_TYPE{"Tipo de índice"}

        FLAT["Flat<br/>No necesita train"]
        TRAIN["IVF / PQ<br/>train con una muestra"]
        HNSW["HNSW<br/>No necesita train"]

        ADD_FLAT["add<br/>Almacena los vectores"]
        ADD_TRAINED["add<br/>Asigna y codifica los vectores"]
        ADD_HNSW["add<br/>Construye el grafo"]

        FAISS[("Índice FAISS<br/>ntotal &gt; 0")]

        CATALOG --> INDEX_TYPE

        INDEX_TYPE -->|Flat| FLAT
        FLAT --> ADD_FLAT
        ADD_FLAT --> FAISS

        INDEX_TYPE -->|IVF / PQ| TRAIN
        TRAIN --> ADD_TRAINED
        ADD_TRAINED --> FAISS

        INDEX_TYPE -->|HNSW| HNSW
        HNSW --> ADD_HNSW
        ADD_HNSW --> FAISS
    end

    subgraph SEARCH["Procesamiento de una consulta"]
        direction LR

        QUERY["Consulta de texto"]
        ENCODER["Mismo encoder<br/>y preprocesamiento"]
        QUERY_VECTOR["Vector de consulta<br/>float32 · dimensión d"]
        SEARCH_FAISS["search(query, k)"]
        CANDIDATES["IDs + scores"]
        METADATA["Agregación<br/>de metadatos"]
        POSTPROCESS["Filtros y reranking"]
        RESULTS["Resultados"]

        QUERY --> ENCODER
        ENCODER --> QUERY_VECTOR
        QUERY_VECTOR --> SEARCH_FAISS
        SEARCH_FAISS --> CANDIDATES
        CANDIDATES --> METADATA
        METADATA --> POSTPROCESS
        POSTPROCESS --> RESULTS
    end

    FAISS --> SEARCH_FAISS

    classDef source fill:#eef2ff,stroke:#4f46e5,color:#1e1b4b;
    classDef decision fill:#fff7ed,stroke:#ea580c,color:#7c2d12;
    classDef process fill:#eff6ff,stroke:#2563eb,color:#172554;
    classDef index fill:#f5f3ff,stroke:#7c3aed,stroke-width:3px,color:#2e1065;
    classDef result fill:#ecfdf5,stroke:#059669,color:#064e3b;

    class CATALOG,QUERY,QUERY_VECTOR source;
    class INDEX_TYPE decision;
    class FLAT,TRAIN,HNSW,ADD_FLAT,ADD_TRAINED,ADD_HNSW,ENCODER,SEARCH_FAISS,CANDIDATES,METADATA,POSTPROCESS process;
    class FAISS index;
    class RESULTS result;
```

### 1.4. Alineación entre filas, IDs y vectores

FAISS asigna por defecto IDs consecutivos según el orden de inserción: el primer vector recibe 0, el segundo 1 y así sucesivamente. Los `product_id` del marketplace son strings y no pueden utilizarse directamente como IDs nativos. La tabla `products` conserva ambos mundos: `vector_id` para FAISS y `product_id` para negocio.

La alineación es un invariante. Si se ordena el DataFrame después de generar la matriz sin aplicar la misma permutación a los vectores, el índice seguirá devolviendo vecinos matemáticamente correctos, pero se mostrarán productos ajenos. Es uno de los fallos más peligrosos porque no produce una excepción.

El cargador valida que `vector_id` sea exactamente `0..N-1`, que las matrices tengan las mismas filas que sus metadatos, que consulta y documentos compartan dimensión y que todas las normas sean aproximadamente uno. Estas comprobaciones deben ocurrir antes de construir cualquier índice.

Vamos a inspeccionar el mapeo `vector_id → product_id` y la distribución de normas. Así confirmaremos que las filas de la matriz pueden traducirse a productos y que el producto escalar implementa realmente el coseno esperado.


In [5]:
products[["vector_id", "product_id", "product_title"]].head()


,vector_id,product_id,product_title
0,0,0007418930,Johnson’s Life of London: The People Who Made ...
1,1,000753101X,Stay With Me: Book 3 (Wait For You)
2,2,0008507775,City on Fire: the gripping new crime novel fro...
3,3,0023544813,General Chemistry: An Integrated Approach
4,4,0029465508,General Chemistry: Principles and Applications


In [6]:
product_norms = np.linalg.norm(product_embeddings, axis=1)
query_norms = np.linalg.norm(query_embeddings, axis=1)

print(f"Normas producto: {product_norms.min():.6f} - {product_norms.max():.6f}")
print(f"Normas consulta: {query_norms.min():.6f} - {query_norms.max():.6f}")


Normas producto: 1.000000 - 1.000000
Normas consulta: 1.000000 - 1.000000


# 2. Búsqueda exacta

## 2.1. k-NN exacto: el oráculo que necesitamos antes de aproximar

Con embeddings normalizados, la similitud coseno coincide con el producto escalar. `IndexFlatIP` almacena los vectores completos en `float32` y, al buscar, calcula el producto interno contra todos. No requiere entrenamiento y no modifica la representación.

El nombre *Flat* no significa que no exista ninguna optimización. FAISS utiliza implementaciones vectorizadas, selección eficiente del top-$k$ y, según la plataforma, instrucciones SIMD o GPU. Lo que no existe es poda algorítmica: todos los candidatos participan en la búsqueda. Por eso garantiza el resultado exacto según los floats almacenados.

Flat cumple dos papeles. Por supuesto, puede ser el índice de producción si el catálogo y el SLA lo permiten. Pero, además, proporciona el ground truth contra el que se calcula el recall de los ANN. Evaluar un índice aproximado sin esta especie de "oráculo" exacto equivale a medir velocidad sin saber qué resultados se han perdido.

A continuación, construiremos nuestro índice `IndexFlatIP`, añadiremos todos los vectores y registraremos tiempo, tamaño y número de elementos. Después buscaremos el top-10 exacto de todo el workload; esos IDs serán el oráculo común de los índices aproximados.


In [7]:
flat_build_started = perf_counter()
flat_index = build_flat_index(product_embeddings)
flat_build_seconds = perf_counter() - flat_build_started

print(f"Entrenado: {flat_index.is_trained}")
print(f"Vectores: {flat_index.ntotal:,}")
print(f"Construcción: {flat_build_seconds:.3f} s")


Entrenado: True
Vectores: 50,000
Construcción: 0.012 s


Nótese que el atributo `is_trained` nos devuelve `True`, pero ya sabemos que este tipo de índices no realiza un entrenamiento como tal.

Seguidamente, realizamos el proceso de búsqueda de nuestras queries:

In [8]:
neighbor_count = 10
exact_scores, exact_ids = flat_index.search(
    np.ascontiguousarray(query_embeddings),
    neighbor_count,
)
print(exact_scores.shape, exact_ids.shape)


(276, 10) (276, 10)


El método `search` en FAISS devuelve dos matrices de forma `(n_queries, k)`: una con los scores obtenidos y otra con los IDs correspondientes. Los scores están ordenados de mayor a menor para el producto escalar. Obviamente, los IDs ocupan la misma posición que su score. Como nota adicional, téngase en cuenta que si el índice contiene menos de $k$ elementos, FAISS completa IDs ausentes con `-1`. Por tanto, ese valor nunca debe utilizarse para indexar un DataFrame porque en pandas o NumPy podría seleccionar accidentalmente la última fila.

Vamos a mostrar a continuación los scores obtenidos para la primera query y los IDs de ese top-10:


In [9]:
print(f"- Consulta realizada: '{queries.iloc[0].query_text}'")
print(f"- 5 primeros valores del embedding de la query: {query_embeddings[0, :5]}")
print(f"- Top-10:")
for score, id in zip(exact_scores[0], exact_ids[0]):
    print(f"\t· Score: {score} | ID: {id}")

- Consulta realizada: 'base tapizada 160x200 sin patas'
- 5 primeros valores del embedding de la query: [ 0.01651075  0.00193472 -0.06214424 -0.07276477  0.02008078]
- Top-10:
	· Score: 0.8897731304168701 | ID: 45992
	· Score: 0.8886998295783997 | ID: 9341
	· Score: 0.8875904083251953 | ID: 30799
	· Score: 0.8874504566192627 | ID: 30828
	· Score: 0.8805969953536987 | ID: 45998
	· Score: 0.8799782991409302 | ID: 8742
	· Score: 0.8794711828231812 | ID: 30729
	· Score: 0.8794504404067993 | ID: 30671
	· Score: 0.8787146806716919 | ID: 6911
	· Score: 0.8786409497261047 | ID: 30706


La siguiente función complementa una consulta concreta. Añade rango y score sin alterar el orden de FAISS. La consulta elegida expresa una necesidad con cambio de unidad: `busco un televisor pequeño de unos setenta centímetros para la cocina`.

Crearemos una función de hidratación que descarte el centinela `-1` y una los IDs devueltos con sus títulos. La aplicaremos a una consulta semántica concreta para leer el ranking como productos, no como dos matrices anónimas.

In [10]:
def hydrate_ranking(
    result_ids: np.ndarray,
    result_scores: np.ndarray,
) -> pd.DataFrame:
    valid_mask = result_ids >= 0
    valid_ids = result_ids[valid_mask]
    ranking = products.iloc[valid_ids][
        ["vector_id", "product_id", "product_title", "product_brand"]
    ].copy()
    ranking.insert(0, "rank", np.arange(1, len(ranking) + 1))
    ranking["score"] = result_scores[valid_mask]
    return ranking


In [11]:
demonstration_position = queries.index[
    queries["workload_id"] == "semantic-101352"
].item()
demonstration_query = queries.iloc[demonstration_position]
print(demonstration_query["query_text"])


busco un televisor pequeño de unas setenta centímetros para la cocina


In [12]:
exact_demonstration = hydrate_ranking(
    exact_ids[demonstration_position],
    exact_scores[demonstration_position],
)
exact_demonstration


,rank,vector_id,product_id,product_title,product_brand,score
16938,1,16938,B075Q9F6J1,"Mantel de tela de algodón y lino de TJW, color...",TJW,0.884411
42059,2,42059,B08KD2RX1J,"SONGMICS Mueble de TV, Armario de TV, Mesa de ...",SONGMICS,0.873443
44349,3,44349,B08T7YZDQ9,TV de red inteligente LED de 32 pulgadas / 42 ...,household items,0.870914
44443,4,44443,B08TLXPSPB,"Home appliances Smart TV 4K UHD con WiFi, Tele...",Home appliances,0.869468
8658,5,8658,B00XOZ1UIY,"SoBuy FRG092-W,Soporte para microondas, Estant...",SoBuy,0.868736
44444,6,44444,B08TM15S1N,"Home appliances Televisores Smart 4K UHD TV, T...",Home appliances,0.868338
33911,7,33911,B07X1XP82C,"Nishore Mesa para TV con 1 Cajón, 1 Estante y ...",Nishore,0.868328
26780,8,26780,B07MM4539M,BONTEC Soporte TV Pie TV Peanas Giratorio Sopo...,BONTEC,0.868226
16917,9,16917,B075NRXVG8,"5 PCS barras extensibles ajustable de 11,8 pul...",HAOYUNTE,0.868041
29123,10,29123,B07Q4GH7D5,"Schneider Consumer - Televisión LED 32"" LED32-...",SCHNEIDER,0.867816


Nótese que, si bien la inmensa mayoría de resultados efectivamente se relacionan con algún producto que sea un televisor, existen otros que no tienen nada que ver con la query dada. Por ejemplo, el resultado que mayor score ha proporcionado es un `Mantel de tela de algodón y lino de TJW, color blanco, diseño tipo daisy con encaje, 100*150cm`.

In [13]:
best_score_product = exact_demonstration.iloc[np.argmax(exact_demonstration.score)].product_title
print(f"Producto con mejor score: '{best_score_product}'")

Producto con mejor score: 'Mantel de tela de algodón y lino de TJW, color blanco, diseño tipo daisy con encaje, 100*150cm'


Es muy probable que el uso de `la cocina` en nuestra query `busco un televisor pequeño de unas setenta centímetros para la cocina` haya desencadenado que un mantel sea el producto más adecuado, dándole un peso dominante a que sea precisamente un producto que pueda usarse en el contexto de una cocina.

Si modificáramos la query a solo `busco un televisor pequeño de unas setenta centímetros` y realizáramos de nuevo el proceso de búsqueda, esto cambiaría.

No obstante, ya de aquí surge una pregunta que, con lo que hemos visto, deberíamos ser capaces de responder: 

* Este comportamiento erróneo, ¿de qué componente de nuestro sistema se deriva? ¿Es culpa del algoritmo de búsqueda, del tratamiento de la información o del modelo de embeddings? ¿Sabrías justificar la respuesta?

## 2.2. Verificar FAISS contra la definición matemática

Como todos los vectores son unitarios, una consulta puede resolverse manualmente mediante `product_embeddings @ query_vector`. Ordenar los scores y tomar los primeros $k$ debe devolver los mismos IDs que `IndexFlatIP`.

Esta prueba es pequeña pero importante. Confirma conjuntamente la métrica, la normalización y la alineación. Si se utilizara `IndexFlatL2` sobre los mismos vectores unitarios, el ranking también coincidiría porque $\lVert q-x\rVert^2=2-2q^\top x$. Los scores, sin embargo, tendrían otra escala y menor sería mejor.


### Calcular scores no es lo mismo que seleccionar top-$k$

La multiplicación exhaustiva produce $N$ scores por consulta, pero la aplicación solo necesita los $k$ mejores. Ordenar completamente $N$ elementos costaría $O(N\log N)$. Las implementaciones eficientes utilizan selección parcial, heaps o kernels fusionados para evitar ordenar la cola irrelevante. Cuando $k$ crece, la selección también se encarece aunque el número de productos comparados no cambie.

El batch modifica el perfil de rendimiento. Una consulta aislada minimiza trabajo total pero aprovecha peor operaciones matriciales y paralelismo. Un lote grande obtiene más throughput, aunque aumenta el tiempo hasta completar todo el batch y puede competir por caché. Por eso `consultas por segundo` y `latencia de una consulta` no son intercambiables.

En servicios online suele aplicarse micro-batching con una ventana muy corta. La capacidad mejora si llegan consultas suficientes, pero la espera de agrupación pasa a formar parte de la latencia extremo a extremo. El benchmark de este notebook usa un lote fijo para comparar estructuras bajo el mismo patrón, no para decidir por sí solo la estrategia de serving.

Calcularemos manualmente los productos escalares de esa misma query, seleccionaremos su top-$k$ y exigiremos que coincida con FAISS. La prueba fijará la definición exacta antes de introducir ninguna aproximación.


In [14]:
manual_scores = product_embeddings @ query_embeddings[demonstration_position]
manual_ids = np.argsort(-manual_scores, kind="stable")[:neighbor_count]

np.testing.assert_array_equal(
    manual_ids,
    exact_ids[demonstration_position],
)
print("El ranking Flat coincide con el producto matricial exhaustivo.")


El ranking Flat coincide con el producto matricial exhaustivo.


## 2.3. El coste crece con el catálogo

La complejidad asintótica no sustituye una medición. El tiempo real depende de ancho de banda de memoria, caché, SIMD, número de threads, tamaño del batch y selección top-$k$. Para aislar el crecimiento con $N$, se fija la dimensión, el hardware, los threads y un lote de 32 consultas.

El benchmark hace un calentamiento, repite la búsqueda y usa la mediana. No incluye el tiempo del encoder, carga desde disco ni hidratación de metadatos. Por tanto, mide exclusivamente la etapa FAISS y no debe presentarse como latencia extremo a extremo del buscador.

Mediremos ahora Flat sobre prefijos crecientes del catálogo manteniendo fijas la dimensión, las consultas y los threads. El gráfico mostrará el escalado observado en esta máquina, sin convertirlo en una cifra universal.


In [15]:
def median_search_ms(
    index: faiss.Index,
    query_matrix: np.ndarray,
    neighbor_limit: int,
    repeats: int = 8,
) -> float:
    index.search(query_matrix[:4], neighbor_limit)
    samples = []
    
    for _ in range(repeats):
        started_at = perf_counter()
        index.search(query_matrix, neighbor_limit)
        samples.append((perf_counter() - started_at) * 1_000)
    
    return float(np.median(samples))


In [16]:
scaling_queries = np.ascontiguousarray(query_embeddings[-32:])
scaling_rows = []

for catalog_size in [1_000, 5_000, 10_000, 25_000, 50_000]:
    sample_index = build_flat_index(product_embeddings[:catalog_size])
    latency_ms = median_search_ms(
        sample_index, scaling_queries, neighbor_count
    )
    scaling_rows.append(
        {
            "products": catalog_size,
            "latency_ms": latency_ms,
            "microseconds_per_query": latency_ms * 1_000 / len(scaling_queries),
        }
    )


In [17]:
scaling_frame = pd.DataFrame(scaling_rows)
scaling_figure = px.line(
    scaling_frame,
    x="products",
    y="latency_ms",
    markers=True,
)
scaling_figure.update_layout(
    title="IndexFlatIP: coste medido al ampliar el catálogo",
    xaxis_title="Productos indexados",
    yaxis_title="Mediana de latencia por lote de 32 consultas (ms)",
)
scaling_figure.show()


Si Flat cumple el SLA con margen, su exactitud, simplicidad y facilidad de actualización son ventajas reales. Introducir ANN en 50.000 productos solo por utilizar una tecnología más sofisticada añadiría entrenamiento, parámetros y modos de fallo.

La necesidad cambia con millones de productos, más consultas concurrentes, dimensiones mayores o presupuestos de CPU estrictos. Para estudiar esa transición sin inventar relevancia, se mantendrá el catálogo real y se observará cuántas comparaciones evita cada estructura.


# 3. Qué significa buscar aproximadamente

Un algoritmo ANN construye una estructura que concentra la búsqueda en una fracción prometedora del espacio. La ganancia aparece si esa fracción es mucho menor que $N$ y puede localizarse sin gastar lo mismo que el barrido completo. La pérdida aparece cuando el verdadero vecino queda fuera de la región explorada o su distancia se estima mediante una representación comprimida.

Las familias principales siguen estrategias diferentes:

- **Partición del espacio.** IVF agrupa vectores alrededor de centroides y examina algunas listas invertidas.
- **Grafos de proximidad.** HNSW navega por enlaces entre vecinos desde capas globales hasta una capa densa.
- **Cuantización.** PQ sustituye floats por códigos compactos y aproxima las distancias mediante tablas.
- **Proyecciones o árboles.** LSH, random projection forests y variantes de k-d trees dividen el espacio mediante hashes o hiperplanos. Su eficacia depende mucho de dimensión y distribución.

FAISS no es un algoritmo único. Es una biblioteca que implementa y compone muchas de estas ideas en CPU y GPU. `IndexIVFPQ`, por ejemplo, combina partición IVF y compresión PQ. La cadena de componentes determina dónde aparece la aproximación.


## 3.1. Por qué los árboles espaciales pierden fuerza en alta dimensión

En dos o tres dimensiones, un k-d tree puede descartar grandes regiones comparando la consulta con planos de corte. En cientos de dimensiones, las regiones se solapan respecto a la vecindad buscada y muchas ramas no pueden podarse con seguridad. El recorrido termina visitando una parte sustancial del árbol y se acerca al coste exhaustivo.

Locality-Sensitive Hashing adopta otra estrategia: diseña funciones hash para que puntos cercanos colisionen con mayor probabilidad. Varias tablas aumentan la oportunidad de colisión, a cambio de memoria y candidatos adicionales. Random projection forests construyen particiones mediante hiperplanos aleatorios. Estas técnicas siguen siendo útiles en ciertos dominios, pero no existe una estructura universal que domine todas las dimensiones, métricas y distribuciones.

IVF aprovecha clustering aprendido del propio corpus; HNSW aprovecha conectividad local; PQ aprovecha redundancia para comprimir. La popularidad de estas familias en embeddings densos no elimina la necesidad de benchmark. Datos muy agrupados, distribuciones anisotrópicas o cambios de modelo pueden alterar por completo su frontera recall-latencia.


## 3.2. Un benchmark ANN necesita un protocolo

Se fijará `k=10` y se utilizarán los 276 vectores de consulta. `exact_ids` contiene el top-10 de Flat. Para cada configuración se registrará:

- recall@10 macro;
- mediana y percentil 95 de latencia por batch;
- consultas por segundo calculadas sobre la mediana;
- tamaño serializado del índice;
- número medio de distancias evaluadas cuando FAISS expone la estadística.

El paralelismo queda fijado porque comparar un índice con un thread contra otro con ocho confunde algoritmo y recursos. El benchmark hace calentamiento para reducir el efecto de carga perezosa y caché fría. Las medidas siguen siendo locales: sirven para comparar configuraciones en esta máquina, no para prometer un SLA en otra.

También se separa construcción de consulta. IVF y PQ necesitan entrenamiento; HNSW construye enlaces costosos al insertar; Flat apenas copia memoria. Un índice de baja latencia puede ser inadecuado si tarda demasiado en reconstruirse ante un catálogo que cambia cada hora.

Prepararemos una estructura de registros común para todos los índices y anotaremos la configuración exacta de Flat. Cada barrido posterior añadirá recall, p50, p95, throughput, construcción y memoria con el mismo esquema.


In [18]:
benchmark_records = []
build_records = []

flat_size_bytes = serialized_size_bytes(flat_index)
flat_result, _ = benchmark_search(
    flat_index,
    query_embeddings,
    exact_ids,
    k=neighbor_count,
    index_name="Flat exacto",
    search_parameter="none",
    parameter_value=0,
    measured_index_size_bytes=flat_size_bytes,
)
benchmark_records.append(flat_result.as_record())
build_records.append(
    {"index": "Flat exacto", "build_seconds": flat_build_seconds}
)


## 4. IVF: localizar primero una región del espacio

Hasta ahora, `IndexFlatIP` comparaba una consulta contra todos los productos del catálogo. Esa estrategia es exacta, pero obliga a recorrer el espacio completo incluso cuando la query solo tiene posibilidades razonables de encontrar buenos candidatos en una zona concreta. IVF introduce una idea muy sencilla: antes de comparar productos, averiguar qué región del espacio merece la pena inspeccionar.

Durante el entrenamiento, IVF aprende `nlist` centroides mediante *k-means*. Cada vector del catálogo se asigna al centroide que le corresponde según la métrica utilizada y se incorpora a la lista invertida asociada. En una geometría euclídea, esos centroides dividen el espacio en **celdas de Voronoi**: una celda contiene todos los puntos que están más cerca de su centroide que de cualquier otro. La lista invertida no es la celda geométrica en sí; es la estructura que guarda los IDs —y, en `IndexIVFFlat`, también los vectores completos— de los productos asignados a ella.

Cuando llega una consulta, IVF no compara directamente su vector con cada producto. Primero compara la query con los centroides, selecciona las `nprobe` celdas más prometedoras y solo recorre las listas almacenadas en ellas. Ahí aparece la aproximación: un vecino relevante puede vivir en una celda que no se ha visitado. Si aumentamos `nprobe`, visitamos más celdas, recuperamos más candidatos potenciales y reducimos ese riesgo, a costa de hacer más trabajo.

En el índice real de esta sesión trabajaremos con embeddings normalizados y producto interno, equivalente a comparar direcciones mediante similitud coseno. La representación bidimensional que construiremos ahora usa distancia euclídea porque permite ver las fronteras con claridad. No pretende reproducir literalmente las 384 dimensiones del catálogo, sino hacer visible el mecanismo que después ocurre en alta dimensión: cada vector queda asignado a una región y la consulta decide qué regiones explorar.

`IndexIVFFlat` conserva los vectores completos dentro de cada lista. Por tanto, una vez elegidas las listas, las comparaciones internas son exactas; la única fuente de aproximación es no visitar todas las celdas. Si `nprobe = nlist`, IVF inspecciona todas las listas y debería converger al mismo ranking que una búsqueda Flat, aunque con el sobrecoste de haber construido y recorrido la estructura IVF.

`nlist` decide cuán fina es la partición del catálogo. Con pocas listas, cada celda contiene demasiados productos y la poda apenas reduce trabajo. Con demasiadas, comparar la query contra todos los centroides cuesta más, hacen falta más datos para entrenar bien y aparecen listas muy pequeñas o inestables. `nprobe`, en cambio, es el mando de consulta: permite modificar el compromiso entre recall y latencia sin reconstruir el índice.

<div align="center">
    <video width="500" autoplay loop muted playsinline>
        <source src="../docs/images/similarity-search-indexes2.mp4" type="video/mp4">
    </video>
</div>

### 4.1. Entrenamiento, asignación y deriva de las listas

El entrenamiento de *k-means* alterna dos movimientos. Primero, cada vector de la muestra escoge el centroide que le corresponde. Después, cada centroide se actualiza a partir de los vectores que le han sido asignados. Al repetir ambos pasos, los centroides terminan describiendo una partición del espacio que intenta aproximarse a la distribución geométrica del catálogo. No aprenden relevancia de negocio, no leen las etiquetas ESCI y no saben qué productos terminarán siendo clicados: únicamente organizan vectores por proximidad.

Una vez terminado el entrenamiento, `add` no vuelve a mover los centroides. Para cada producto nuevo, calcula a qué celda pertenece y lo introduce en la lista invertida correspondiente. Esto permite añadir productos sin reconstruir el índice, pero también explica la deriva: si el marketplace incorpora una categoría nueva, cambia el encoder o modifica sustancialmente la distribución de sus textos, los centroides siguen representando la muestra con la que se entrenaron originalmente.

En ese caso pueden aparecer listas descompensadas. Una lista enorme concentra coste de búsqueda; una lista vacía o diminuta puede indicar que `nlist` es excesivo, que la muestra de entrenamiento era insuficiente o que la distribución no se ha particionado de forma estable. La deriva no se diagnostica observando solo los centroides: se contrasta el tamaño de las listas, la distancia de los vectores a su centroide, la distribución de consultas por celda y el recall frente a un índice Flat construido sobre un snapshot reciente.

Antes de trabajar con los embeddings de 384 dimensiones, construiremos una partición bidimensional de juguete. No nos limitaremos a colorear los puntos según su lista: dibujaremos también las celdas de Voronoi inducidas por los centroides aprendidos. Así podremos distinguir visualmente los productos asignados a cada lista de la frontera geométrica que determina esa asignación.

In [19]:
toy_generator = np.random.default_rng(7)

toy_centers = np.array(
    [
        [-2, -1],
        [2, -1],
        [-1, 2],
        [2, 2],
    ],
    dtype=np.float32,
)

toy_points = np.vstack(
    [
        center + toy_generator.normal(scale=0.55, size=(60, 2))
        for center in toy_centers
    ]
).astype(np.float32)

toy_kmeans = faiss.Kmeans(
    d=2,
    k=4,
    niter=30,
    seed=42,
    verbose=False,
)
toy_kmeans.train(toy_points)

toy_centroids = toy_kmeans.centroids
_, toy_assignments = toy_kmeans.index.search(toy_points, 1)
toy_assignments = toy_assignments.ravel()

In [20]:
all_toy_coordinates = np.vstack([toy_points, toy_centroids])

x_minimum, y_minimum = all_toy_coordinates.min(axis=0) - 0.6
x_maximum, y_maximum = all_toy_coordinates.max(axis=0) + 0.6

x_grid = np.linspace(x_minimum, x_maximum, 350)
y_grid = np.linspace(y_minimum, y_maximum, 350)

grid_x, grid_y = np.meshgrid(x_grid, y_grid)
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

squared_distances = (
    (grid_points[:, np.newaxis, :] - toy_centroids[np.newaxis, :, :]) ** 2
).sum(axis=2)

voronoi_cell_ids = squared_distances.argmin(axis=1).reshape(grid_x.shape)

In [21]:
cell_colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]

voronoi_colorscale = [
    [0.00, cell_colors[0]],
    [0.2499, cell_colors[0]],
    [0.25, cell_colors[1]],
    [0.4999, cell_colors[1]],
    [0.50, cell_colors[2]],
    [0.7499, cell_colors[2]],
    [0.75, cell_colors[3]],
    [1.00, cell_colors[3]],
]

toy_figure = go.Figure()

toy_figure.add_trace(
    go.Heatmap(
        x=x_grid,
        y=y_grid,
        z=voronoi_cell_ids,
        zmin=0,
        zmax=3,
        colorscale=voronoi_colorscale,
        opacity=0.18,
        showscale=False,
        hoverinfo="skip",
    )
)

for list_id, color in enumerate(cell_colors):
    points_in_cell = toy_points[toy_assignments == list_id]

    toy_figure.add_trace(
        go.Scatter(
            x=points_in_cell[:, 0],
            y=points_in_cell[:, 1],
            mode="markers",
            marker={"size": 9, "color": color},
            name=f"Lista {list_id}",
        )
    )

toy_figure.add_trace(
    go.Scatter(
        x=toy_centroids[:, 0],
        y=toy_centroids[:, 1],
        mode="markers",
        marker={"symbol": "x", "size": 19, "color": "black"},
        name="Centroides",
    )
)

toy_figure.update_layout(
    title="Las listas de IVF corresponden a celdas de Voronoi",
    xaxis_title="Dimensión 1",
    yaxis_title="Dimensión 2",
    template="plotly_white",
    height=680,
    margin={"l": 70, "r": 190, "t": 90, "b": 70},
)

toy_figure.update_xaxes(
    range=[x_minimum, x_maximum],
    constrain="domain",
)

toy_figure.update_yaxes(
    range=[y_minimum, y_maximum],
    scaleanchor="x",
    scaleratio=1,
    constrain="domain",
)

toy_figure.show()

Véase que en la frontera entre celdas puede surgir un fallo típico en este tipo de algoritmos: un vecino real puede quedar en la segunda celda más cercana a la query aunque esté muy cerca geométricamente. Esto haría que con `nprobe=1` nunca fuera considerado. Por tanto, aumentar `nprobe` reduce ese error de frontera.

Para los 50.000 vectores, utilizaremos 256 listas y 30.000 vectores de entrenamiento (es una configuración de ejemplo, no son parámetros universales ni están específicamente optimizados para el ejemplo tratado). FAISS propone equilibrar el coste de comparar centroides y escanear listas mediante la regla aproximada de situar `nlist` alrededor de un múltiplo de $\sqrt{N}$, pero la selección final debe surgir de la curva recall-latencia.

Entrenaremos a continuación `IndexIVFFlat` con 256 listas sobre la muestra de 30.000 y añadiremos los 50.000 productos. Inspeccionaremos el tamaño de las listas antes de barrer `nprobe`, porque visitar ocho listas no equivale a visitar ocho listas iguales.

<div align="center">
    <img src="../docs/images/ivf-nprobe-1.webp" width=500 height=300>
</div>

<div align="center">
    <img src="../docs/images/ivf-nprobe-8.webp" width=500 height=300>
</div>


In [22]:
ivf_build_started = perf_counter()
ivf_index = build_ivf_flat_index(
    product_embeddings,
    nlist=256,
    training_size=30_000,
    seed=42,
)
ivf_build_seconds = perf_counter() - ivf_build_started

print(f"Entrenado: {ivf_index.is_trained}")
print(f"Vectores: {ivf_index.ntotal:,}")
print(f"Construcción: {ivf_build_seconds:.2f} s")


Entrenado: True
Vectores: 50,000
Construcción: 0.15 s


In [23]:
ivf_list_sizes = np.array(
    [ivf_index.invlists.list_size(list_id) for list_id in range(ivf_index.nlist)]
)
pd.Series(ivf_list_sizes).describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
)


count    256.000000
mean     195.312500
std       90.409634
min        9.000000
5%        63.500000
25%      130.000000
50%      193.000000
75%      257.000000
95%      352.000000
max      471.000000
dtype: float64

In [24]:
list_figure = px.histogram(
    x=ivf_list_sizes,
    nbins=30,
    labels={"x": "Productos por lista", "y": "Número de listas"},
    title="El k-means no produce listas perfectamente equilibradas",
)
list_figure.show()


El desequilibrio importa porque `nprobe=8` no implica examinar exactamente $8/256$ del catálogo. Algunas listas contienen más productos y ciertas consultas caen en regiones densas. Las estadísticas internas de FAISS permiten observar el número real de distancias calculadas.

El índice se barre ahora con varios valores de `nprobe`. La estructura y el tamaño serializado no cambian; solo cambia el presupuesto de exploración de cada consulta.

Ejecutaremos el barrido de `nprobe` manteniendo fijo el índice. Para cada valor mediremos cuántas distancias calcula FAISS, qué recall conserva y qué latencia paga; así podremos relacionar la geometría de las listas con la curva resultante.


In [25]:
ivf_size_bytes = serialized_size_bytes(ivf_index)
ivf_sweep_records = []
ivf_rankings = {}
for nprobe in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    ivf_index.nprobe = nprobe
    result, result_ids = benchmark_search(
        ivf_index,
        query_embeddings,
        exact_ids,
        k=neighbor_count,
        index_name="IVF-Flat",
        search_parameter="nprobe",
        parameter_value=nprobe,
        measured_index_size_bytes=ivf_size_bytes,
    )
    ivf_sweep_records.append(result.as_record())
    ivf_rankings[nprobe] = result_ids


In [26]:
ivf_frame = pd.DataFrame(ivf_sweep_records)
benchmark_records.extend(ivf_sweep_records)
build_records.append(
    {"index": "IVF-Flat", "build_seconds": ivf_build_seconds}
)
ivf_frame[
    [
        "value",
        "recall_at_k",
        "median_latency_ms",
        "distance_computations_per_query",
    ]
]


,value,recall_at_k,median_latency_ms,distance_computations_per_query
0,1,0.411957,1.312479,229.427536
1,2,0.536232,1.695812,459.076087
2,4,0.647101,2.725875,922.992754
3,8,0.773188,4.906125,1812.949275
4,16,0.862319,9.420125,3543.242754
5,32,0.934420,18.355104,6908.670290
6,64,0.971014,40.291855,13532.014493
7,128,0.995652,85.683271,26621.148551
8,256,1.000000,170.485271,50000.000000


In [27]:
ivf_figure = px.line(
    ivf_frame,
    x="median_latency_ms",
    y="recall_at_k",
    text="value",
    markers=True,
)
ivf_figure.update_traces(textposition="top center")
ivf_figure.update_layout(
    title="IVF-Flat: nprobe desplaza el punto recall-latencia",
    xaxis_title="Mediana de latencia por batch (ms)",
    yaxis_title="Recall@10 contra Flat",
)
ivf_figure.show()


### 4.2. Inspeccionar una consulta que IVF pierde

El recall medio resume el comportamiento del índice, pero puede ocultar historias muy distintas. Un valor razonable de `recall@10` puede convivir con consultas para las que IVF recupera prácticamente el mismo ranking que Flat y con otras para las que no aparece ni uno solo de los diez vecinos exactos. Antes de decidir que un valor de `nprobe` es aceptable, conviene mirar alguno de esos casos extremos y entender qué ha ocurrido.

Con `nprobe=1`, IVF solo visita la celda cuyo centroide considera más cercano a la query. Si los vecinos que devolvería una búsqueda Flat están repartidos entre esa celda y otras regiones próximas, los productos almacenados fuera de la única lista visitada ni siquiera llegan a competir. No se trata de que IVF puntúe peor a los candidatos que sí ha inspeccionado: `IndexIVFFlat` conserva los vectores completos y calcula sus scores de forma exacta. La pérdida aparece antes, cuando la poda decide qué listas quedan fuera de la búsqueda.

Calcularemos ahora `recall@10` para cada consulta del workload con `nprobe=1` y localizaremos la que obtiene el valor más bajo. Si varias empatan en el mínimo, `argmin` escogerá la primera; no buscamos demostrar que sea una query especial por su texto, sino disponer de un ejemplo concreto en el que observar la consecuencia de visitar una única celda.

Después enfrentaremos los IDs devueltos por Flat con los obtenidos por IVF. La tabla mostrará ambos rankings por posición y señalará si cada candidato aproximado pertenece al top-10 exacto, aunque aparezca en un rango distinto. Si el recall es bajo, veremos que los vecinos perdidos no han recibido un score bajo: sencillamente nunca estuvieron entre los productos candidatos porque su lista no formaba parte de la búsqueda.

In [28]:
ivf_low_recall = recall_per_query(
    exact_ids,
    ivf_rankings[1],
    k=neighbor_count,
)
worst_ivf_position = int(np.argmin(ivf_low_recall))
print(queries.iloc[worst_ivf_position]["query_text"])
print(f"Recall@10: {ivf_low_recall[worst_ivf_position]:.2f}")


convertibles 2 en 1 portátil tactil
Recall@10: 0.00


In [29]:
pd.DataFrame(
    {
        "rank": np.arange(1, neighbor_count + 1),
        "exact_id": exact_ids[worst_ivf_position],
        "ivf_nprobe_1_id": ivf_rankings[1][worst_ivf_position],
        "coincide": np.isin(
            ivf_rankings[1][worst_ivf_position],
            exact_ids[worst_ivf_position],
        ),
    }
)


,rank,exact_id,ivf_nprobe_1_id,coincide
0,1,40959,10287,False
1,2,41416,34373,False
2,3,37670,29643,False
3,4,48406,5261,False
4,5,9859,5360,False
5,6,7413,32087,False
6,7,42617,21869,False
7,8,39161,38376,False
8,9,41751,10444,False
9,10,46397,4272,False


# 5. HNSW: navegar por un grafo de proximidad

**Hierarchical Navigable Small World (HNSW)** aborda la búsqueda aproximada desde una idea distinta a la de IVF. En lugar de dividir el espacio en celdas y decidir cuáles visitar, organiza los vectores como nodos de un grafo y conecta cada uno con otros elementos cercanos. La búsqueda deja así de parecerse a una exploración de particiones y pasa a comportarse como un recorrido: partimos de un nodo conocido y avanzamos por enlaces que nos acercan progresivamente a la query.

Ese grafo no tiene una única capa. En la capa inferior aparecen todos los vectores y se concentra la conectividad local, mientras que las capas superiores contienen subconjuntos cada vez más pequeños. Los nodos que alcanzan esas capas altas funcionan como puntos de paso de largo alcance: permiten atravesar rápidamente regiones amplias del espacio antes de descender al nivel donde se encuentran los detalles.

Una consulta comienza en un punto de entrada situado en la capa más alta disponible. Desde allí, HNSW examina los vecinos del nodo actual y se desplaza de forma greedy hacia aquel que mejora la distancia respecto a la query. Cuando ningún vecino ofrece una mejora, el algoritmo baja una capa y continúa desde la posición alcanzada. Cada descenso aumenta el nivel de detalle: las capas superiores sirven para aproximarse a la zona correcta y la capa cero se encarga de refinar la búsqueda entre conexiones mucho más locales.

La analogía habitual es una skip list. No porque ambas estructuras resuelvan exactamente el mismo problema, sino porque comparten la misma intuición: unos pocos enlaces de gran alcance evitan recorrer paso a paso toda la estructura, mientras que los enlaces cortos permiten precisar el resultado al acercarnos al destino. La jerarquía reduce así el riesgo de comenzar a explorar a ciegas un grafo formado por todos los elementos.

La aproximación aparece porque el recorrido no examina el grafo completo. Las decisiones greedy pueden conducir a una región que parece prometedora localmente, pero desde la que no se alcanza alguno de los verdaderos vecinos más próximos. Además, la exploración en la capa cero está limitada por `efSearch`, de modo que incluso una ruta razonable puede detenerse antes de descubrir mejores candidatos. Aumentar la conectividad o ampliar la búsqueda reduce ese riesgo, aunque a cambio de más memoria, mayor coste de construcción o más latencia por consulta.

<div align="center">
    <img src="../docs/images/hnsw-algorithm.webp" width=600 height=400>
</div>

## 5.1. Cómo se construye la jerarquía

La jerarquía de HNSW no surge de entrenar centroides ni de aprender una partición global del espacio. Cada nuevo vector recibe aleatoriamente un nivel máximo, y la probabilidad de alcanzar capas altas decrece de forma exponencial. Como consecuencia, todos los elementos aparecen en la capa cero, una fracción menor alcanza la capa uno y solo unos pocos llegan a los niveles superiores. Esa distribución es la que crea las autopistas del grafo sin necesidad de una fase previa de entrenamiento.

Cuando se inserta un elemento, el algoritmo comienza en el punto de entrada actual y desciende por la jerarquía buscando una región próxima al nuevo vector. En las capas superiores el recorrido sirve sobre todo para aproximarse rápidamente a una buena zona. Al llegar a las capas en las que el nuevo nodo debe quedar conectado, la inserción amplía la exploración, reúne un conjunto de candidatos y decide cuáles conservar como vecinos.

Esa selección no consiste siempre en tomar sin más los $M$ puntos de menor distancia. Si todos los enlaces apuntaran hacia nodos casi idénticos entre sí, el vecindario sería muy preciso en una sola dirección, pero poco útil para desplazarse por el resto del espacio. Por eso HNSW emplea una heurística de diversidad que puede descartar algún candidato muy cercano para mantener conexiones hacia regiones distintas. El objetivo no es únicamente representar la proximidad inmediata, sino construir rutas que permitan escapar de zonas localmente atractivas durante búsquedas futuras.

Los enlaces suelen establecerse en ambas direcciones. Cuando la inserción hace que un nodo supere su capacidad, su lista de vecinos se vuelve a seleccionar y se poda. Este proceso es incremental: cada elemento se incorpora sobre el grafo existente y no hay una optimización global posterior que revise todas las conexiones una vez terminada la construcción.

Ahí aparece la importancia de `efConstruction`. Este parámetro determina cuántos candidatos se consideran durante la inserción. Con una exploración amplia, el nuevo nodo tiene más oportunidades de descubrir conexiones útiles y diversas. Con una exploración demasiado estrecha, puede quedar enlazado a una región poco representativa, y esa decisión pasa a formar parte de la estructura. Aumentar después `efSearch` puede compensar parcialmente un grafo mediocre mediante una búsqueda más exhaustiva, pero no reconstruye los enlaces que nunca llegaron a crearse.

Gracias a la distribución aleatoria de niveles, la altura esperada de la jerarquía crece de forma logarítmica y las capas superiores ofrecen saltos cada vez más largos. Esa propiedad explica la eficiencia habitual de HNSW, pero no garantiza por sí sola una latencia concreta para cualquier conjunto de datos. La geometría de los embeddings, el tamaño del índice y los parámetros de construcción siguen condicionando el comportamiento real, por lo que la medición empírica continúa siendo imprescindible.

<div align="center">
    <img src="../docs/images/hnsw-index-vectors.webp" width=500 height=300>
</div>

## 5.2. Los tres controles principales

El comportamiento de HNSW depende sobre todo de tres parámetros que intervienen en momentos distintos. `M` y `efConstruction` determinan cómo se forma el grafo; `efSearch` decide cuánto se explora ese grafo cuando llega una consulta. Separar esos papeles resulta importante porque no todos los ajustes pueden corregirse online.

`M` limita aproximadamente el número de vecinos que cada nodo conserva. Con un valor bajo, el grafo ocupa menos memoria y las inserciones son más baratas, pero existen menos rutas alternativas entre regiones. Un `M` mayor crea una estructura más conectada, lo que suele mejorar el recall, especialmente cuando la geometría del espacio es compleja o presenta muchas direcciones relevantes. Esa mejora no es gratuita: cada nodo almacena más enlaces y la construcción debe evaluar y mantener vecindarios más amplios.

`efConstruction` controla la amplitud de la búsqueda utilizada al insertar cada elemento. Cuantos más candidatos se exploran, más información tiene el algoritmo para elegir conexiones cercanas y, al mismo tiempo, suficientemente diversas. El resultado suele ser un grafo de mayor calidad, aunque la construcción tarda más. No es un parámetro de consulta: una vez creado el índice, modificar `efConstruction` no revisa las conexiones existentes ni repara una estructura construida con una exploración insuficiente.

`efSearch`, en cambio, actúa durante cada búsqueda. En la capa cero, HNSW mantiene candidatos pendientes y mejores resultados encontrados; `efSearch` limita aproximadamente el tamaño de esa exploración. Un valor pequeño reduce la latencia, pero puede detener el recorrido antes de alcanzar alguna región relevante. Un valor mayor permite considerar más rutas y suele aumentar el recall. Debe ser al menos tan grande como $k$, aunque ese requisito solo establece un mínimo operativo y no garantiza una calidad determinada.

En ese sentido, `efSearch` desempeña un papel parecido a `nprobe` en IVF. Ambos permiten desplazarse por una curva de compromiso entre recall y latencia sin reconstruir el índice. Sin embargo, no existe un valor universalmente correcto: la elección depende del dataset, del número de vecinos solicitados, de la distribución de las consultas y del SLA que se quiera cumplir.

Trabajaremos con `IndexHNSWFlat`, que conserva los vectores completos. Cuando un nodo entra en el conjunto de candidatos visitados, su distancia respecto a la query se calcula sobre el vector original, sin cuantización. Por tanto, la pérdida de recall no procede de aproximar los scores, sino de la navegación: algunos nodos nunca llegan a visitarse porque el recorrido del grafo no alcanza su región o porque el presupuesto de exploración se agota antes.

Esta precisión en las distancias no significa que el índice tenga el mismo coste de memoria que Flat. A los aproximadamente $4d$ bytes necesarios para almacenar cada vector en `float32` hay que añadir los enlaces del grafo, la información de niveles y la estructura auxiliar de navegación. HNSW intercambia memoria por velocidad y calidad de recuperación.

Para aislar el efecto de la búsqueda, construiremos un único grafo con valores fijos de `M` y `efConstruction`. Mediremos primero el coste de esa construcción y, sobre la misma estructura, barreremos distintos valores de `efSearch`. De este modo, cualquier variación de recall o latencia procederá de explorar con mayor o menor amplitud el mismo grafo, y no de comparar índices construidos con conectividades distintas.

In [30]:
hnsw_build_started = perf_counter()
hnsw_index = build_hnsw_index(
    product_embeddings,
    graph_degree=24,
    ef_construction=120,
)
hnsw_build_seconds = perf_counter() - hnsw_build_started

print(f"Vectores: {hnsw_index.ntotal:,}")
print(f"M: {hnsw_index.hnsw.nb_neighbors(0) // 2}")
print(f"efConstruction: {hnsw_index.hnsw.efConstruction}")
print(f"Construcción: {hnsw_build_seconds:.2f} s")


Vectores: 50,000
M: 24
efConstruction: 120
Construcción: 2.87 s


In [48]:
hnsw_size_bytes = serialized_size_bytes(hnsw_index)

hnsw_sweep_records = []
hnsw_rankings = {}

for ef_search in [10, 16, 24, 32, 48, 64, 96, 128, 192, 256]:
    hnsw_index.hnsw.efSearch = ef_search
    result, result_ids = benchmark_search(
        hnsw_index,
        query_embeddings,
        exact_ids,
        k=neighbor_count,
        index_name="HNSW-Flat",
        search_parameter="efSearch",
        parameter_value=ef_search,
        measured_index_size_bytes=hnsw_size_bytes,
    )
    hnsw_sweep_records.append(result.as_record())
    hnsw_rankings[ef_search] = result_ids


In [49]:
hnsw_frame = pd.DataFrame(hnsw_sweep_records)
benchmark_records.extend(hnsw_sweep_records)
build_records.append(
    {"index": "HNSW-Flat", "build_seconds": hnsw_build_seconds}
)
hnsw_frame[
    [
        "value",
        "recall_at_k",
        "median_latency_ms",
        "distance_computations_per_query",
    ]
]


,value,recall_at_k,median_latency_ms,distance_computations_per_query
0,10,0.710507,3.043605,408.061594
1,16,0.783333,3.970250,530.898551
2,24,0.836232,4.859313,688.688406
3,32,0.867391,6.140792,838.188406
4,48,0.906884,8.095104,1124.829710
5,64,0.930435,10.004542,1389.952899
6,96,0.955072,14.686458,1886.652174
7,128,0.967029,17.577958,2345.115942
8,192,0.980072,24.374771,3186.250000
9,256,0.984783,30.826521,3978.007246


In [50]:
hnsw_figure = px.line(
    hnsw_frame,
    x="median_latency_ms",
    y="recall_at_k",
    text="value",
    markers=True,
)
hnsw_figure.update_traces(textposition="top center")
hnsw_figure.update_layout(
    title="HNSW: efSearch controla cuánto se navega",
    xaxis_title="Mediana de latencia por batch (ms)",
    yaxis_title="Recall@10 contra Flat",
)
hnsw_figure.show()


## 5.3. Construcción, mutaciones y memoria

HNSW puede incorporar nuevos elementos después de haber creado el índice, pero una inserción no consiste simplemente en añadir un vector al final de una matriz. Cada alta debe recorrer la jerarquía, localizar una región próxima, explorar candidatos y decidir con qué nodos quedará conectada. Por tanto, el coste de actualización depende del propio grafo y de los parámetros elegidos para construirlo.

Además, como la construcción es incremental, el estado del grafo en el momento de cada inserción influye en las conexiones disponibles. Dos índices creados con los mismos vectores y los mismos parámetros pueden no resultar idénticos si cambia el orden de llegada o la aleatoriedad con la que se asignan los niveles. En conjuntos razonables, ambos pueden ofrecer un comportamiento parecido, pero conviene recordar que HNSW no produce una estructura única determinada exclusivamente por los datos.

Las eliminaciones son más problemáticas. Un nodo no solo representa un producto almacenado: también puede formar parte de las rutas que permiten alcanzar otros nodos. Retirarlo físicamente exige revisar conexiones entrantes y salientes, y puede degradar la navegabilidad del grafo si la reparación no se realiza con cuidado. Por ese motivo, la implementación HNSW de FAISS no ofrece una operación general de borrado equivalente a eliminar una fila de un índice Flat.

En sistemas con bajas frecuentes, una estrategia habitual consiste en separar la validez lógica de los productos de la estructura física del índice. El nodo permanece temporalmente en el grafo, pero su ID se marca como inactivo en una capa externa y se filtra antes de devolver los resultados. Cuando se acumulan suficientes altas, bajas o cambios de distribución, se construye un nuevo índice a partir del catálogo vigente y se sustituye el anterior.

Esto obliga a evaluar HNSW como parte de un ciclo de vida completo. Para un marketplace no basta con medir cuántos milisegundos tarda una consulta sobre un índice ya cargado. También importan el tiempo de construcción inicial, el coste de incorporar novedades, el porcentaje de elementos inactivos tolerable, la frecuencia de reconstrucción, la creación de snapshots, el despliegue paralelo de una nueva versión y la posibilidad de volver atrás si el nuevo artefacto empeora recall o latencia.

Mediremos también el tamaño serializado del índice. Esa cifra permite observar cuánto añaden los enlaces y los niveles respecto a un Flat que almacena los mismos vectores, y comparar el coste persistente con el de IVF. No debe interpretarse como una medida exacta de la memoria residente del proceso: el RSS puede incluir buffers temporales, fragmentación del allocator, objetos auxiliares y páginas mapeadas. Aun así, el tamaño del artefacto guardado es una métrica reproducible y útil para comparar configuraciones bajo las mismas condiciones.

# 6. Product Quantization: comprimir para poder buscar

Hasta ahora hemos utilizado índices que conservan cada embedding completo. Con 384 dimensiones en `float32`, un producto necesita 384 valores de cuatro bytes, es decir, 1.536 bytes antes de añadir IDs, listas, enlaces u otras estructuras del índice. Esta cantidad parece modesta para un único vector, pero escala rápidamente: diez millones de productos requieren alrededor de 15 GB solo para las coordenadas.

Cuando la memoria pasa a ser la restricción principal, ya no basta con reducir el número de candidatos examinados. IVF y HNSW pueden evitar comparar la query contra todo el catálogo, pero sus variantes Flat siguen almacenando los vectores originales. Product Quantization introduce una aproximación distinta: en lugar de conservar cada coordenada, aprende una representación compacta con la que estima las distancias durante la búsqueda.

La idea consiste en dividir el vector en $m$ bloques o subvectores. Cada bloque ocupa un subespacio de menor dimensión y dispone de su propio conjunto de centroides, denominado codebook. Durante el entrenamiento, PQ aprende $2^{nbits}$ centroides para cada uno de esos subespacios. Después, cada fragmento de un producto se sustituye por el identificador del centroide que mejor lo representa.

El vector deja así de almacenarse como una secuencia de floats. En su lugar, se guarda una sucesión de códigos discretos: uno por subespacio. La representación ya no describe exactamente dónde estaba el producto, sino qué centroide se eligió para aproximar cada parte de su embedding.

Con `m=48` y `nbits=8`, las 384 dimensiones se dividen en 48 subvectores de ocho dimensiones. Cada subvector puede elegir entre 256 centroides, y esa elección cabe en un byte. El código completo ocupa aproximadamente 48 bytes por producto frente a los 1.536 bytes del vector original. Antes de contabilizar codebooks, IDs y la estructura IVF, la reducción teórica es de 32 veces.

La búsqueda tampoco necesita reconstruir explícitamente los 384 valores aproximados de cada candidato. PQ utiliza normalmente Asymmetric Distance Computation, o ADC. La query permanece sin comprimir y, para cada uno de sus subvectores, se calculan las distancias a todos los centroides del codebook correspondiente. Esas distancias se guardan en pequeñas tablas de consulta.

Cuando se evalúa un producto, sus códigos indican qué entrada debe leerse en cada tabla. La distancia total se aproxima sumando esas entradas. Este procedimiento sustituye una comparación completa entre dos vectores `float32` por una serie de accesos y sumas sobre una representación mucho más compacta.

La ventaja es que el índice necesita muchos menos bytes por producto y puede mantener más datos en memoria. El coste es que la distancia deja de ser exacta. Dos productos cercanos pueden compartir códigos parecidos, y pequeñas diferencias presentes en los embeddings originales pueden desaparecer al sustituir cada subvector por su centroide.

<div align="center">
    <video width="500" autoplay loop muted playsinline>
        <source src="../docs/images/product-quantization-6.mp4" type="video/mp4">
    </video>
</div>

## 6.1. Los parámetros `m` y `nbits` cambian capacidad y coste

La calidad de Product Quantization depende en gran medida de cómo se reparten los bits disponibles entre los distintos subespacios. Los parámetros `m` y `nbits` determinan cuántos fragmentos tendrá el vector y cuántas representaciones posibles puede utilizar cada fragmento. Juntos controlan tanto el tamaño del código como la cantidad de información que puede conservarse.

Si el embedding tiene dimensión $d=384$ y elegimos `m=48`, cada subvector contiene ocho coordenadas. Cada codebook debe resumir, por tanto, un espacio de ocho dimensiones. Si aumentamos `m`, los bloques se vuelven más pequeños y cada codebook tiene que modelar relaciones menos complejas. Esto suele reducir la distorsión, porque un centroide debe resumir menos información conjunta.

La contrapartida es que aparece un código adicional por cada nuevo subespacio. Con ocho bits por código, pasar de 48 a 64 subcuantizadores aumenta de 48 a 64 bytes la representación de cada producto. Un `m` menor comprime más, pero obliga a cada codebook a representar bloques de mayor dimensión, donde un número limitado de centroides puede resultar insuficiente para capturar toda la variabilidad.

En la configuración habitual de FAISS, la dimensión debe ser divisible por `m`, porque todos los subvectores tienen el mismo tamaño. No cualquier valor es válido para un embedding de 384 dimensiones: `m=48` produce bloques de ocho coordenadas, mientras que `m=32` produciría bloques de doce.

`nbits` determina cuántos centroides existen en cada codebook. Con `nbits=8`, cada subespacio dispone de $2^8=256$ centroides y el identificador elegido ocupa un byte. Aumentar el número de bits amplía el vocabulario disponible para representar cada fragmento y puede reducir el error de cuantización, pero también incrementa el tamaño de los codebooks, el esfuerzo de entrenamiento y el almacenamiento necesario para los códigos.

Reducir `nbits` produce una compresión más agresiva. Con cuatro bits solo existen 16 centroides por subespacio, pero dos códigos pueden empaquetarse en un mismo byte. El índice ocupa menos memoria, aunque cada fragmento tiene muchas menos alternativas para aproximar su posición original.

Formalmente, dividimos la query $q$ en subvectores $q_j$. Un producto queda representado por una secuencia de índices $a_j$, donde cada índice selecciona un centroide $c_{j,a_j}$ del codebook correspondiente. ADC aproxima entonces la distancia mediante

$$
\widetilde{d}(q,x)=\sum_{j=1}^{m} d(q_j,c_{j,a_j}).
$$

Para una query concreta, primero se construye una tabla con la distancia entre cada $q_j$ y todos los centroides de su subespacio. Evaluar un producto consiste después en leer, para cada bloque, la entrada señalada por su código y sumar los valores obtenidos. No es necesario ejecutar de nuevo una operación sobre las 384 coordenadas originales de cada candidato.

Esta reducción de cálculo y memoria no implica que la latencia vaya a mejorar siempre en la misma proporción que el tamaño. El rendimiento depende también del formato de los códigos, de la vectorización SIMD, del tamaño de las tablas, de los accesos a memoria y del número de candidatos examinados. En colecciones pequeñas, la lógica adicional de decodificación puede incluso dominar. La ventaja de PQ aparece con mayor claridad cuando el volumen de datos y el movimiento de memoria se convierten en el verdadero cuello de botella.

## 6.2. IVF-PQ combina dos fuentes de aproximación

`IndexIVFPQ` combina la poda de candidatos de IVF con la compresión de Product Quantization. La partición gruesa decide primero en qué lista se almacena cada producto y qué listas se visitan durante una consulta. PQ se encarga después de representar de forma compacta los vectores contenidos en esas listas.

En lugar de cuantizar directamente el embedding completo, IVF-PQ suele codificar el residual respecto al centroide de la lista. Si un producto $x$ ha sido asignado al centroide grueso $c$, el código PQ representa aproximadamente la diferencia $x-c$. El centroide IVF ya explica en qué región general del espacio se encuentra el producto; PQ solo necesita describir cómo se separa de ese punto de referencia.

Esta descomposición suele funcionar mejor que cuantizar todos los vectores con un único sistema global. Los residuales tienden a estar más concentrados alrededor del origen y contienen menos estructura de gran escala. Cada codebook puede dedicar así su capacidad a modelar variaciones locales dentro de una celda, en lugar de representar simultáneamente regiones muy alejadas del espacio.

La combinación introduce, sin embargo, dos oportunidades distintas para perder vecinos. La primera aparece en la fase IVF: si la lista de un vecino exacto no está entre las seleccionadas por `nprobe`, ese producto nunca llega a evaluarse. La segunda aparece después, dentro de las listas visitadas: aunque el candidato sí participe en la búsqueda, su distancia se calcula a partir del código PQ y puede quedar ordenado por detrás de otros productos cuya aproximación resulte más favorable.

Esta diferencia es importante al interpretar una curva de recall. Aumentar `nprobe` permite visitar más listas y reduce las pérdidas causadas por la partición gruesa. Pero no modifica los códigos almacenados ni hace más precisas las distancias cuantizadas. Si el recall mejora al principio y luego se estabiliza aunque se estén explorando muchas listas, el límite ya no procede principalmente de la poda IVF, sino de la resolución de PQ.

En ese punto, seguir aumentando `nprobe` añade trabajo sin recuperar necesariamente los vecinos perdidos. Para elevar el techo de calidad habría que enriquecer la representación: aumentar `m`, utilizar más bits por subcuantizador o aplicar una transformación como OPQ que reorganice las dimensiones antes de dividirlas en bloques. Otra opción consiste en utilizar IVF-PQ para generar un conjunto amplio de candidatos y rerankear después los mejores con los vectores originales, siempre que esos embeddings se conserven en memoria o en un almacenamiento secundario suficientemente rápido.

La calidad del índice también depende del entrenamiento. Cada uno de los $m$ codebooks debe aprender sus centroides a partir de ejemplos representativos. Si la muestra es demasiado pequeña, algunos centroides apenas reciben observaciones y el espacio queda modelado de manera deficiente. FAISS puede emitir avisos cuando el número de vectores no resulta suficiente para entrenar de forma razonable el número solicitado de centroides.

No basta, además, con utilizar muchos ejemplos si proceden de una distribución distinta. La muestra de entrenamiento debe parecerse al catálogo que se indexará y, en la medida de lo posible, a los datos que llegarán después. Un codebook aprendido con una categoría dominante puede comprimir mal productos pertenecientes a regiones apenas representadas.

Construiremos un `IndexIVFPQ` con la misma partición gruesa de 256 listas utilizada anteriormente, 48 subcuantizadores y ocho bits por código. Después variaremos `nprobe` sin cambiar el índice. Al comparar la nueva curva con la de IVF-Flat podremos separar ambos efectos: la mejora inicial mostrará cuánto recall se recupera visitando más listas, mientras que una posible meseta revelará la pérdida que permanece incluso cuando los candidatos sí han entrado en la búsqueda y sus distancias se evalúan mediante PQ.

In [34]:
pq_build_started = perf_counter()
ivf_pq_index = build_ivf_pq_index(
    product_embeddings,
    nlist=256,
    subquantizers=48,
    bits_per_code=8,
    training_size=30_000,
    seed=42,
)
pq_build_seconds = perf_counter() - pq_build_started

print(f"Code size: {ivf_pq_index.code_size} bytes por producto")
print(f"Construcción: {pq_build_seconds:.2f} s")


Code size: 48 bytes por producto
Construcción: 1.45 s


In [35]:
pq_size_bytes = serialized_size_bytes(ivf_pq_index)
pq_sweep_records = []
pq_rankings = {}
for nprobe in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    ivf_pq_index.nprobe = nprobe
    result, result_ids = benchmark_search(
        ivf_pq_index,
        query_embeddings,
        exact_ids,
        k=neighbor_count,
        index_name="IVF-PQ",
        search_parameter="nprobe",
        parameter_value=nprobe,
        measured_index_size_bytes=pq_size_bytes,
    )
    pq_sweep_records.append(result.as_record())
    pq_rankings[nprobe] = result_ids


In [36]:
pq_frame = pd.DataFrame(pq_sweep_records)
benchmark_records.extend(pq_sweep_records)
build_records.append(
    {"index": "IVF-PQ", "build_seconds": pq_build_seconds}
)
pq_frame[
    [
        "value",
        "recall_at_k",
        "median_latency_ms",
        "distance_computations_per_query",
    ]
]


,value,recall_at_k,median_latency_ms,distance_computations_per_query
0,1,0.269203,2.181687,229.427536
1,2,0.333333,2.304354,459.076087
2,4,0.367754,2.565645,922.992754
3,8,0.404348,3.193729,1812.949275
4,16,0.427174,4.300542,3543.242754
5,32,0.435507,6.078479,6908.670290
6,64,0.439130,9.884792,13532.014493
7,128,0.441667,17.121229,26621.148551
8,256,0.442029,31.463209,50000.000000


In [37]:
pq_figure = px.line(
    pq_frame,
    x="median_latency_ms",
    y="recall_at_k",
    text="value",
    markers=True,
)
pq_figure.update_traces(textposition="top center")
pq_figure.update_layout(
    title="IVF-PQ: nprobe no elimina el error de cuantización",
    xaxis_title="Mediana de latencia por batch (ms)",
    yaxis_title="Recall@10 contra Flat",
)
pq_figure.show()


Si el recall se aplana antes de uno, visitar más listas ya no basta. La distancia aproximada de PQ ha cambiado el orden de algunos candidatos. Esta diferencia muestra por qué `ANN` no es una sola causa de error y por qué un único parámetro no siempre puede recuperar calidad.

Existen cuantizadores escalares que codifican cada coordenada por separado, variantes PQ aceleradas, OPQ que rota el espacio antes de dividirlo y cuantizadores aditivos. La familia correcta depende del presupuesto de memoria y del recall objetivo. En este material se mantiene una configuración comprensible para que cada pérdida tenga una causa identificable.


# 7. Comparar configuraciones: la frontera de Pareto

Después de medir varias familias de índices y barrer parámetros como `nprobe` o `efSearch`, es tentador reducir la comparación a una clasificación simple: cuál ha sido el más rápido, cuál ha alcanzado el mayor recall y cuál ocupa menos memoria. El problema es que esos tres títulos suelen recaer en configuraciones distintas. Una opción puede ser muy rápida porque sacrifica demasiados vecinos, otra puede acercarse a Flat a costa de una latencia inaceptable y una tercera puede comprimir bien los datos, pero quedarse por debajo del nivel de calidad exigido.

La decisión real no consiste, por tanto, en encontrar un ganador absoluto, sino en identificar qué configuraciones cumplen las restricciones del sistema. Podríamos exigir, por ejemplo, un `recall@10` de al menos 0,95, un tamaño compatible con la memoria disponible y una latencia p95 que no supere el SLA del servicio. Una configuración que no satisface alguno de esos límites deja de ser candidata, aunque destaque en las demás métricas.

La frontera de Pareto permite ordenar el resto sin imponer todavía una única función de coste. Diremos que una configuración está dominada cuando existe otra que ofrece un recall igual o mayor con una latencia igual o menor. En ese caso, la primera no aporta ninguna ventaja dentro de esas dos dimensiones: siempre habría una alternativa al menos tan buena en calidad y en velocidad.

Los puntos que no están dominados forman la frontera de Pareto. Cada uno representa un compromiso distinto. En un extremo aparecen configuraciones muy rápidas con menor recall; en el otro, búsquedas más exhaustivas que recuperan más vecinos a cambio de mayor latencia. Ningún punto de la frontera es universalmente mejor que los demás: la elección depende de cuánto valor se conceda a cada milisegundo adicional y de qué nivel mínimo de fidelidad necesite la siguiente etapa.

Modificar `nprobe` en IVF o `efSearch` en HNSW suele desplazar una misma estructura a lo largo de esa curva. Al aumentar la exploración, la latencia crece y el recall tiende a mejorar. Cambiar de familia de índice produce un efecto más profundo: altera la forma de la curva, el coste de construcción, la memoria necesaria y, en algunos casos, el techo máximo de recall alcanzable.

El gráfico siguiente situará todas las mediciones en el plano recall-latencia e incluirá Flat como referencia exacta. El tamaño de cada punto representará el tamaño serializado del índice, de modo que una configuración aparentemente atractiva por velocidad y calidad también revele cuánto cuesta mantenerla en memoria o persistirla.

La frontera obtenida no debe interpretarse como una propiedad universal de cada algoritmo. Sus posiciones dependen del hardware, del número de threads, del tamaño del batch, de la distribución de los vectores y de las consultas, así como de la versión concreta de FAISS. El gráfico resume este experimento bajo unas condiciones determinadas; cambiar cualquiera de ellas puede desplazar los puntos e incluso alterar qué configuraciones resultan dominadas.

## 7.1. Recall depende de $k$ y de la distribución de consultas

El recall no es una propiedad única del índice. Siempre está definido para un valor concreto de $k$, y cambiar ese valor puede modificar de forma sustancial la impresión que obtenemos de una configuración. Un índice puede recuperar casi siempre el vecino más próximo y mostrar un `recall@1` excelente, pero perder una fracción mucho mayor cuando se le pide reconstruir el top-100 exacto.

Una razón habitual es que el primer vecino se encuentra claramente separado del resto, mientras que en posiciones más profundas aparecen muchos candidatos con distancias muy parecidas. En esa zona del ranking, una pequeña diferencia provocada por la poda del índice o por la cuantización puede intercambiar posiciones y hacer que un elemento entre o salga del conjunto recuperado. Cuanto mayor es $k$, más oportunidades existen para que esas pequeñas desviaciones afecten al recall.

Por eso $k$ no debería elegirse como una convención arbitraria del benchmark. Debe reflejar cuántos candidatos consume realmente la siguiente etapa del sistema. Si un reranker solo recibe 20 productos, `recall@20` resulta más informativo que `recall@100`. Si la búsqueda ANN alimenta una fase posterior que necesita 200 candidatos para mantener diversidad o aplicar reglas de negocio, evaluar únicamente `recall@10` puede ocultar pérdidas relevantes.

La forma de agregar el resultado también importa. La media macro asigna el mismo peso a cada consulta y ofrece un resumen fácil de comparar, pero no muestra cómo se distribuyen los errores. Dos configuraciones pueden compartir un recall medio de 0,95 y comportarse de manera muy distinta: una puede rondar ese valor en casi todas las consultas, mientras que otra combina muchas búsquedas perfectas con un grupo pequeño de fallos extremos.

Conviene, por tanto, acompañar la media con percentiles, mínimos y análisis por slices. Los peores casos pueden concentrarse en consultas raras, categorías con alta densidad, embeddings ambiguos o regiones que estuvieron poco representadas durante el entrenamiento de IVF o PQ. Un promedio sólido puede convivir con un segmento concreto en el que el índice no recupera ninguno de los vecinos exactos.

Nuestro workload combina consultas de negocio con 256 queries de prueba construidas a partir de títulos. Las consultas de negocio conservan formulaciones literales y paráfrasis que se parecen al uso previsto del sistema. Las queries, en cambio, amplían la cobertura geométrica del espacio y fuerzan al benchmark a visitar regiones que quizá no aparecerían en un conjunto pequeño de búsquedas manuales.

Esas queries sirven para medir fidelidad respecto a Flat, pero no contienen juicios humanos de relevancia. No sabemos si todos los vecinos exactos son igualmente útiles desde el punto de vista del usuario, solo que son los que devuelve la métrica vectorial elegida. Por ello pueden utilizarse para estimar recall ANN, pero no para calcular métricas como nDCG, que necesitan conocer qué resultados son realmente relevantes y, en muchos casos, con qué grado.

En producción, la fuente principal del workload deberían ser los logs reales de búsqueda, tratados con las garantías de privacidad correspondientes. Ese muestreo debe conservar la diversidad de consultas y también una dimensión temporal. Si entrenamos, ajustamos y evaluamos sobre el mismo snapshot, corremos el riesgo de elegir parámetros demasiado adaptados a una distribución concreta y no detectar cómo cambian las consultas o el catálogo con el tiempo.

Una partición temporal permite observar esa deriva. Los parámetros pueden ajustarse con consultas de un periodo y evaluarse sobre otro posterior, reproduciendo mejor la situación en la que un índice construido hoy tendrá que atender tráfico futuro. Esta separación no elimina todos los sesgos, pero reduce la posibilidad de optimizar el benchmark en lugar del sistema real.

Reuniremos finalmente todas las mediciones en un único `DataFrame`. Cada fila describirá una configuración concreta e incluirá recall, latencia, tamaño serializado y tiempo de construcción. Sobre esa tabla dibujaremos el plano recall-latencia, diferenciando la familia de índice y utilizando el tamaño del punto para representar la memoria.

De este modo, una curva rápida no podrá ocultar que necesita un artefacto muy grande o una construcción especialmente costosa. La comparación mantendrá juntas las métricas online y offline, porque la mejor configuración no es solo la que responde bien a una consulta, sino la que puede construirse, desplegarse, actualizarse y mantenerse dentro de las restricciones completas del sistema.

In [38]:
benchmark_frame = pd.DataFrame(benchmark_records)
benchmark_frame["index_size_mib"] = (
    benchmark_frame["index_size_bytes"] / 2**20
)
benchmark_frame["configuration"] = (
    benchmark_frame["parameter"]
    + "="
    + benchmark_frame["value"].astype(str)
)


In [39]:
pareto_figure = px.scatter(
    benchmark_frame,
    x="median_latency_ms",
    y="recall_at_k",
    color="index",
    size="index_size_mib",
    hover_data=[
        "configuration",
        "p95_latency_ms",
        "queries_per_second",
        "distance_computations_per_query",
    ],
    size_max=42,
)
pareto_figure.update_layout(
    title="Recall, latencia y memoria de todas las configuraciones",
    xaxis_title="Mediana de latencia por batch (ms)",
    yaxis_title="Recall@10 contra Flat",
    height=620,
)
pareto_figure.show()


In [40]:
size_frame = (
    benchmark_frame.groupby("index", as_index=False)["index_size_mib"]
    .first()
    .sort_values("index_size_mib")
)
size_figure = px.bar(
    size_frame,
    x="index",
    y="index_size_mib",
    text_auto=".1f",
    title="Tamaño serializado de cada estructura",
)
size_figure.update_layout(yaxis_title="MiB")
size_figure.show()


In [41]:
build_frame = pd.DataFrame(build_records)
build_figure = px.bar(
    build_frame,
    x="index",
    y="build_seconds",
    text_auto=".2f",
    title="El coste de construcción también forma parte de la decisión",
)
build_figure.update_layout(yaxis_title="Segundos")
build_figure.show()


## 7.2. Seleccionar bajo una restricción explícita

Una vez dibujada la frontera de Pareto, todavía queda tomar una decisión. El gráfico muestra qué compromisos son eficientes, pero no determina por sí solo qué punto debe desplegarse. Para elegir hace falta introducir una política concreta: qué nivel mínimo de fidelidad estamos dispuestos a aceptar y, entre las configuraciones que lo alcanzan, qué coste queremos minimizar.

En este caso fijaremos como requisito un `recall@10` de al menos 0,95. A partir de ahí descartaremos todas las configuraciones que no cumplan el umbral y, entre las restantes, elegiremos la que presente la menor mediana de latencia. El procedimiento es sencillo, pero tiene una ventaja importante: transforma una preferencia ambigua por “ir lo más rápido posible sin perder demasiada calidad” en una regla explícita, verificable y reproducible.

El valor 0,95 no debe interpretarse como una recomendación universal. Es una decisión de producto y de riesgo. Un sistema médico puede exigir una fidelidad mucho mayor porque omitir un candidato relevante tiene un coste elevado. Un proceso offline de deduplicación quizá pueda explorar durante más tiempo para acercarse a recall perfecto. Un carrusel comercial, en cambio, puede tolerar pequeñas diferencias si la latencia adicional perjudica la experiencia de navegación. El umbral correcto depende del uso, no del algoritmo.

También conviene separar recall técnico de relevancia para el usuario. `recall@10` mide cuántos elementos del top-10 exacto de Flat aparecen en el resultado aproximado. Si el décimo vecino exacto es irrelevante para la intención de búsqueda, recuperarlo mejora la métrica ANN, pero no necesariamente la experiencia. Del mismo modo, una configuración puede perder algún vecino exacto y aun así devolver productos igual de útiles desde el punto de vista del negocio.

Por eso esta selección solo valida la fidelidad del índice respecto a la búsqueda exacta. La evaluación del sistema completo debe incorporar además métricas de relevancia como nDCG, resultados por categorías o tipos de consulta y señales online como conversión, abandono o interacción. El índice ANN es una pieza del pipeline, no el objetivo final.

Aplicaremos, por tanto, una regla deliberadamente clara: conservar únicamente las configuraciones con `recall@10 >= 0.95` y ordenarlas por mediana de latencia. La primera fila resultante será nuestra mejor opción bajo esa restricción concreta. Si el umbral cambia, la decisión puede cambiar también; precisamente por eso interesa expresar la política en código y no dejarla implícita en una inspección visual del gráfico.

In [42]:
recall_threshold = 0.95
eligible_configurations = benchmark_frame.loc[
    benchmark_frame["recall_at_k"] >= recall_threshold
].sort_values("median_latency_ms")
eligible_configurations[
    [
        "index",
        "configuration",
        "recall_at_k",
        "median_latency_ms",
        "p95_latency_ms",
        "index_size_mib",
    ]
].head(10)


,index,configuration,recall_at_k,median_latency_ms,p95_latency_ms,index_size_mib
16,HNSW-Flat,efSearch=96,0.955072,16.087271,18.901602,83.173872
17,HNSW-Flat,efSearch=128,0.967029,23.951896,33.189460,83.173872
18,HNSW-Flat,efSearch=192,0.980072,25.037479,29.320234,83.173872
19,HNSW-Flat,efSearch=256,0.984783,32.490021,39.217175,83.173872
7,IVF-Flat,nprobe=64,0.971014,40.291855,42.861058,74.000743
8,IVF-Flat,nprobe=128,0.995652,85.683271,88.475110,74.000743
0,Flat exacto,none=0,1.000000,127.967854,163.493779,73.242230
9,IVF-Flat,nprobe=256,1.000000,170.485271,178.526090,74.000743


# 8. Metadatos, filtros y persistencia

## 8.1. FAISS solo conoce vectores e IDs

En un marketplace, la similitud vectorial rara vez basta para decidir qué productos pueden aparecer en una respuesta. Una consulta puede estar limitada a artículos disponibles en stock, vendidos en un país concreto, pertenecientes a determinadas categorías o marcas, situados dentro de un rango de precios o visibles únicamente para ciertos usuarios. El índice ANN puede encontrar productos semánticamente cercanos, pero no conoce por sí mismo ninguna de esas reglas.

FAISS trabaja esencialmente con vectores y con los IDs que permiten identificar cada elemento. No se comporta como una base de datos documental capaz de almacenar columnas de negocio y evaluar expresiones como `stock > 0`, `country = "ES"` o `price < 50` durante cualquier búsqueda. Por eso, la capa vectorial suele convivir con otro sistema responsable de los metadatos, ya sea una base de datos, un motor de búsqueda o una caché especializada.

El patrón más directo consiste en pedir al índice más de $k$ candidatos, recuperar después los metadatos asociados a sus IDs y descartar aquellos que no cumplen las restricciones. Si queremos devolver diez productos, por ejemplo, podemos solicitar cien vecinos al índice, hidratar sus atributos y conservar los diez primeros que superen el filtro. Este enfoque se conoce como **post-filter**, porque la condición se aplica después de la recuperación vectorial.

Su principal problema aparece cuando el filtro es selectivo. Si solo una pequeña parte de los cien candidatos tiene stock o pertenece al país solicitado, el sistema puede terminar devolviendo menos de los diez resultados esperados. Aumentar el *oversampling* reduce ese riesgo: en lugar de recuperar cien candidatos, podríamos recuperar quinientos o mil. Sin embargo, cada ampliación obliga al índice a explorar más resultados, aumenta la transferencia de IDs y encarece la hidratación de metadatos. Además, ningún factor fijo garantiza llenar el cupo cuando la condición acepta una fracción extremadamente pequeña del catálogo.

La alternativa es aplicar la restricción antes o durante la búsqueda. En un **pre-filter**, otro componente calcula primero el conjunto de IDs permitidos y la recuperación vectorial solo puede considerar esos elementos. Desde el punto de vista lógico, este procedimiento evita que candidatos inválidos consuman posiciones del top-$k$. Su eficiencia, sin embargo, depende de cómo pueda utilizar el índice ese conjunto de IDs.

Si la estructura ANN no admite el filtro de forma nativa, obtener los IDs permitidos no basta: habría que comparar la query únicamente contra ese subconjunto mediante otro mecanismo o mantener índices separados para distintas poblaciones. Algunas variantes IVF de FAISS permiten utilizar selectores de IDs y excluir candidatos durante el recorrido de las listas. Esta posibilidad puede ser útil, aunque el coste seguirá dependiendo de cuántos elementos de cada lista sean válidos y de cómo se represente el selector.

Otra estrategia consiste en mantener un índice independiente por segmento, como un país, una gran categoría o un nivel de permisos. Así, la consulta se dirige desde el principio a una colección donde todos los elementos cumplen parte del filtro. El enfoque funciona bien cuando existen pocos segmentos estables y con suficiente volumen, pero se complica si los filtros tienen alta cardinalidad o pueden combinarse libremente. Crear un índice para cada país, marca, categoría, rango de precio y combinación entre ellos produciría una proliferación difícil de construir, actualizar y desplegar.

La arquitectura adecuada depende, por tanto, de la forma real de los filtros. Importan su selectividad, su cardinalidad, cuántas condiciones suelen combinarse, la frecuencia con la que cambian los metadatos y el coste de devolver menos de $k$ resultados. Los filtros amplios y poco frecuentes pueden resolverse mediante post-filter con cierto oversampling; los segmentos estables pueden justificar índices separados; y las restricciones dinámicas o muy selectivas pueden requerir selectores nativos o una integración más estrecha entre el motor vectorial y la capa de metadatos.

### Elegir entre post-filter, selectores y particiones

No existe una única estrategia de filtrado que funcione igual de bien para todos los catálogos. La elección depende de cuántos candidatos elimina cada condición, de la rapidez con la que pueden obtenerse los IDs permitidos y de cómo esté organizado el índice. En la práctica, post-filter, selectores y particiones representan tres formas distintas de decidir en qué momento se reduce el universo de búsqueda.

El **post-filter** mantiene un único índice y separa con claridad la recuperación vectorial de las reglas de negocio. El índice devuelve sus vecinos más próximos, la aplicación recupera los metadatos asociados y descarta después los productos que no cumplen el filtro. Este patrón resulta sencillo y eficaz cuando la condición elimina pocos candidatos, porque basta con pedir algunos resultados adicionales para completar el top-$k$ final.

El comportamiento cambia cuando el filtro es muy selectivo. Para mostrar diez productos de una marca concreta puede ser necesario recuperar cientos o miles de vecinos globales antes de encontrar diez que pertenezcan a ella. La latencia deja entonces de depender únicamente de $k$ y pasa a depender de una tasa de aceptación que puede variar mucho entre consultas. Además, el índice ANN ha optimizado su recorrido para encontrar el top-$k$ del catálogo completo, no el top-$k$ dentro del subconjunto permitido. Los mejores elementos de ese subconjunto pueden encontrarse mucho más abajo en el ranking global o no aparecer siquiera dentro del presupuesto de exploración utilizado.

Los **selectores de ID** intentan introducir la restricción durante la propia búsqueda. La capa de metadatos produce un conjunto, bitmap o estructura equivalente con los IDs permitidos, y FAISS descarta los demás mientras examina candidatos. De esta forma, un producto inválido no ocupa una posición provisional que después será eliminada por la aplicación.

Este enfoque resulta especialmente atractivo cuando el conjunto permitido puede construirse rápidamente y el tipo de índice sabe utilizarlo de manera eficiente. Sin embargo, filtrar candidatos no implica necesariamente evitar todo el trabajo asociado a ellos. Si el índice debe recorrer listas llenas de productos no permitidos para descubrir que casi ninguno pasa el selector, la búsqueda puede seguir siendo costosa. La utilidad real depende de la estructura concreta, de la API empleada y de si la restricción permite omitir regiones completas o solo rechazar elementos una vez inspeccionados. La compatibilidad no es uniforme entre todas las familias de índices de FAISS.

La tercera opción consiste en **particionar el catálogo** antes de construir los índices. Podemos mantener, por ejemplo, un índice por país o por gran categoría. La consulta se dirige entonces a una colección donde todos los productos cumplen ya una parte de las restricciones, lo que reduce el universo de búsqueda y permite ajustar parámetros distintos para cada segmento.

Esta separación puede ser muy útil cuando existen pocas particiones estables y cada una conserva suficiente volumen para entrenar y evaluar bien su índice. También permite desplegar o reconstruir un país sin afectar necesariamente a los demás. El problema aparece cuando la segmentación se vuelve demasiado fina. Muchas particiones pequeñas aumentan el número de artefactos que deben entrenarse, cargarse, versionarse y monitorizarse, y algunas pueden contener tan pocos ejemplos que IVF o PQ aprendan estructuras poco fiables.

Las combinaciones de filtros complican todavía más el diseño. Un índice por país no resuelve por sí solo una consulta que también restringe marca, precio y categoría. Crear una partición para cada combinación posible no escala, y los productos que pertenecen a varios segmentos pueden terminar duplicados en distintos índices, aumentando memoria y coste de actualización.

Una base de datos vectorial suele envolver estas decisiones en una plataforma más amplia. Además de la estructura ANN, incorpora almacenamiento de metadatos, evaluación de filtros, persistencia, replicación, concurrencia y mecanismos de actualización. FAISS se concentra principalmente en los algoritmos y estructuras de búsqueda. Cuando se utiliza directamente, corresponde a la aplicación coordinar el índice con la base de metadatos, decidir cómo se propagan las altas y bajas, garantizar que ambos sistemas comparten la misma versión y diseñar la estrategia de durabilidad y recuperación.

Para observar el comportamiento más sencillo, simularemos un filtro de marca mediante post-filter sobre Flat. Pediremos al índice más vecinos de los que finalmente queremos mostrar, utilizaremos sus IDs para recuperar la marca de cada producto y descartaremos aquellos que no pertenezcan al conjunto permitido. Al variar el nivel de oversampling podremos ver cuándo aparecen suficientes candidatos para completar el top-$k$ y cuándo, pese a ampliar la recuperación, el resultado sigue quedándose corto.

In [43]:
filter_scores, filter_ids = flat_index.search(
    np.ascontiguousarray(
        query_embeddings[demonstration_position : demonstration_position + 1]
    ),
    500,
)
filter_candidates = products.iloc[filter_ids[0]].copy()
common_brands = filter_candidates["product_brand"].value_counts()
target_brand = common_brands.index[0]
print(f"Filtro de ejemplo: product_brand == {target_brand!r}")


Filtro de ejemplo: product_brand == 'LG'


In [44]:
filter_rows = []
for retrieval_depth in [10, 25, 50, 100, 250, 500]:
    candidate_ids = filter_ids[0, :retrieval_depth]
    candidate_products = products.iloc[candidate_ids]
    matching_count = int(
        candidate_products["product_brand"].eq(target_brand).sum()
    )
    filter_rows.append(
        {
            "retrieval_depth": retrieval_depth,
            "matching_results": min(matching_count, 10),
        }
    )


In [45]:
filter_frame = pd.DataFrame(filter_rows)
filter_figure = px.line(
    filter_frame,
    x="retrieval_depth",
    y="matching_results",
    markers=True,
    title="Post-filter: recuperar más no siempre llena el top-10",
)
filter_figure.update_layout(
    xaxis_title="Candidatos recuperados antes del filtro",
    yaxis_title="Resultados que cumplen la marca (máximo 10)",
)
filter_figure.show()


El ejemplo selecciona deliberadamente una marca que ya aparece en la vecindad de la consulta para que el mecanismo pueda observarse con claridad. Así resulta más fácil seguir cómo el oversampling amplía el conjunto inicial, cómo se hidratan los metadatos y en qué momento aparecen suficientes productos válidos para completar el top-$k$.

En un sistema real, sin embargo, la marca no se elige para facilitar la demostración. Procede de la petición del usuario y puede representar desde una fracción grande del catálogo hasta un subconjunto casi residual. Esa selectividad puede variar varios órdenes de magnitud entre filtros, categorías y mercados, de modo que un factor de oversampling que funciona bien en una consulta puede resultar insuficiente o excesivo en otra.

Por ese motivo, no basta con medir el recall global del índice antes de aplicar restricciones. La métrica relevante es el recall condicionado al filtro: cuántos de los mejores vecinos exactos que además cumplen la condición aparecen finalmente en la respuesta. Un índice puede mostrar una fidelidad global excelente y, aun así, comportarse mal para una marca minoritaria si sus productos quedan demasiado dispersos en el ranking general.

También es necesario decidir en qué punto del pipeline se aplican las reglas de negocio. Algunas condiciones, como excluir productos sin stock o sin permisos de visualización, son restricciones duras: un resultado que no las cumple no debería llegar nunca al usuario. Otras transformaciones, como diversificar marcas, favorecer popularidad o limitar el número de productos de una misma categoría, actúan más bien como reglas de reordenación sobre un conjunto ya recuperado.

Cada una de esas etapas modifica el ranking por una razón distinta. La búsqueda ANN puede perder candidatos por aproximación; el filtro puede eliminar productos válidos semánticamente pero no elegibles; y una regla posterior puede desplazar resultados para mejorar diversidad o cumplir objetivos comerciales. Si todas esas transformaciones se observan únicamente a través del ranking final, resulta difícil saber dónde se originó una degradación.

La evaluación debe mantener, por tanto, métricas separadas para la recuperación vectorial, el filtrado y el reordenamiento de negocio. Solo así podremos distinguir si una consulta devuelve pocos resultados porque el índice no encontró suficientes candidatos, porque el filtro era extremadamente selectivo o porque una regla posterior alteró deliberadamente el orden visible.

## 8.2. Persistencia, carga y ciclo de vida

Un índice que funciona correctamente dentro de un notebook todavía no constituye un despliegue. Mientras vive únicamente en memoria, desaparece al terminar el proceso y no puede reproducirse ni distribuirse con garantías. Para convertirlo en un artefacto operativo hay que guardarlo, identificar con precisión cómo fue construido y asegurar que puede cargarse después sin cambiar su comportamiento.

FAISS permite serializar un índice mediante `write_index` y reconstruirlo posteriormente con `read_index`. El fichero resultante contiene la estructura necesaria para ejecutar búsquedas, pero por sí solo no explica qué representa. Para interpretar correctamente sus vectores hacen falta también el modelo de embeddings, la dimensión, la métrica utilizada, la política de normalización, la plantilla con la que se generó el texto, el snapshot de datos y todos los parámetros de construcción.

Esos elementos forman una única versión lógica. Un índice creado con embeddings normalizados para producto interno no puede reutilizarse de forma segura con queries sin normalizar. Del mismo modo, cambiar el modelo, la plantilla de texto o el orden de las columnas puede producir vectores incompatibles aunque la dimensión siga siendo la misma. Versionar únicamente el archivo `.index` deja fuera parte del estado que determina el resultado de la búsqueda.

La tabla de metadatos debe pertenecer al mismo snapshot. FAISS devuelve IDs, y otra capa los traduce a productos, marcas, precios o disponibilidad. Si se publica un índice nuevo mientras la base de metadatos conserva una versión anterior, un mismo ID puede apuntar al producto equivocado, no existir o referirse a atributos ya obsoletos. El índice y sus metadatos deben desplegarse como una unidad coherente.

Una estrategia segura construye la nueva versión en paralelo sin reemplazar todavía la que atiende tráfico. Sobre ese artefacto se ejecutan comprobaciones de integridad, búsquedas de referencia, métricas de recall, validaciones de tamaño y controles de correspondencia entre IDs y metadatos. Solo cuando esas pruebas terminan correctamente se cambia un alias, una ruta o un puntero para dirigir las consultas hacia la nueva versión.

Ese cambio debería ser atómico. No conviene que algunas instancias utilicen el índice nuevo mientras consultan metadatos antiguos, ni que una petición vea una versión distinta a mitad del proceso. Mantener la versión anterior durante un tiempo permite además realizar rollback si aparecen regresiones de calidad, latencia o estabilidad que no fueron detectadas durante las pruebas previas.

La carga también exige considerar la seguridad del artefacto. `read_index` no debería utilizarse sobre archivos de procedencia desconocida o no confiable. Un índice de FAISS es una estructura binaria compleja que el proceso interpreta directamente; no debe tratarse como un formato inocuo para aceptar entradas arbitrarias. Antes de cargarlo conviene verificar su procedencia, permisos, tamaño esperado y checksum.

El checksum permite detectar corrupción accidental o sustituciones inesperadas durante almacenamiento y transferencia. No demuestra por sí solo que el contenido sea seguro, pero sí confirma que el fichero cargado coincide con el artefacto que se publicó. Combinado con un registro de versiones y una fuente de confianza, forma parte de una cadena mínima de integridad.

Cerraremos el ciclo escribiendo el índice HNSW a disco y cargándolo después en una instancia nueva. Ejecutaremos sobre ambas versiones el mismo conjunto de consultas y compararemos los primeros rankings devueltos. La igualdad de IDs no valida todos los aspectos de un despliegue, pero sí proporciona una prueba mínima de que la serialización y la carga han conservado la estructura de búsqueda sin alterar sus resultados.

In [46]:
artifact_directory = project_root / ".artifacts" / "indexes"
artifact_directory.mkdir(parents=True, exist_ok=True)
hnsw_path = artifact_directory / "hnsw_flat.faiss"

hnsw_index.hnsw.efSearch = 96
faiss.write_index(hnsw_index, str(hnsw_path))
reloaded_hnsw = faiss.read_index(str(hnsw_path))
reloaded_hnsw.hnsw.efSearch = 96


In [47]:
_, original_hnsw_ids = hnsw_index.search(
    np.ascontiguousarray(query_embeddings[:8]), neighbor_count
)
_, reloaded_hnsw_ids = reloaded_hnsw.search(
    np.ascontiguousarray(query_embeddings[:8]), neighbor_count
)
np.testing.assert_array_equal(original_hnsw_ids, reloaded_hnsw_ids)
print(f"Índice recargado correctamente desde {hnsw_path.name}")


Índice recargado correctamente desde hnsw_flat.faiss


# 9. Decisión para el marketplace

Después de comparar varias familias de índices, la conclusión no tiene por qué ser que el marketplace necesite búsqueda aproximada. Con un catálogo de 50.000 productos, una búsqueda Flat puede ser suficientemente rápida y ofrece una ventaja difícil de ignorar: devuelve el ranking exacto bajo la métrica elegida, admite inserciones sin una estructura compleja y reduce al mínimo el número de componentes que hay que construir, versionar y mantener.

Si Flat cumple el SLA cuando se prueba con el hardware real, el volumen esperado de consultas, la concurrencia de producción y los filtros habituales, utilizarlo no representa una solución provisional ni una falta de sofisticación. Es la opción más sencilla que satisface los requisitos. Introducir ANN antes de necesitarlo añadiría aproximación y complejidad operativa sin una ganancia demostrada.

La situación cambia cuando crece el catálogo, aumenta la carga o la latencia de Flat deja de ser compatible con el servicio. En ese momento, las curvas medidas durante el notebook permiten elegir con más criterio que una comparación basada únicamente en reputación o popularidad de los algoritmos. Cada estructura resuelve una restricción distinta y desplaza el coste hacia una parte diferente del sistema.

**IVF-Flat** resulta atractivo cuando se acepta una fase de entrenamiento y se quiere controlar de forma explícita qué fracción del catálogo se examina. `nprobe` permite ampliar o reducir el número de listas visitadas y ofrece un mecanismo intuitivo para recorrer la curva recall-latencia. Como los vectores permanecen completos, las distancias de los candidatos inspeccionados son exactas; la pérdida aparece porque parte del catálogo queda fuera de la búsqueda.

Esa claridad tiene un coste operativo. Los centroides representan un snapshot de la distribución y pueden perder calidad si el catálogo cambia con el tiempo. También hay que vigilar si las listas quedan desequilibradas, porque dos configuraciones con el mismo `nprobe` pueden terminar inspeccionando cantidades de productos muy distintas. IVF-Flat no solo exige medir la consulta: obliga a observar la salud de la partición y a decidir cuándo merece la pena volver a entrenarla.

**HNSW-Flat** suele ofrecer una combinación especialmente competitiva de recall y latencia. La jerarquía permite acercarse rápidamente a una región prometedora y refinar después la búsqueda en el grafo completo. Además, el índice puede incorporar nuevos elementos sin reconstruirse desde cero, lo que lo hace atractivo para catálogos con altas continuas.

La contrapartida es que esa navegabilidad se paga con memoria adicional y con una construcción más costosa. Los enlaces del grafo ocupan espacio, su calidad depende de `M` y `efConstruction`, y las bajas no encajan de forma tan natural como las inserciones. Adoptar HNSW implica diseñar también una política para IDs inactivos, reconstrucciones periódicas, despliegues paralelos y rollback.

**IVF-PQ** responde a un problema diferente. Su principal motivación no es únicamente reducir latencia, sino hacer que un catálogo grande quepa dentro del presupuesto de memoria. Al sustituir los vectores originales por códigos compactos, puede reducir de forma drástica los bytes almacenados por producto y permitir que conjuntos mucho mayores permanezcan en memoria.

Esa compresión introduce una segunda fuente de aproximación. IVF puede omitir la lista correcta y PQ puede alterar el orden incluso entre candidatos que sí fueron visitados. Por eso aumentar `nprobe` no garantiza acercarse indefinidamente a Flat: llega un punto en el que el techo de recall está marcado por la cuantización. Elegir IVF-PQ supone aceptar esa pérdida a cambio de una reducción de memoria que, en determinados volúmenes, puede ser la diferencia entre un sistema viable y otro que no cabe en la infraestructura disponible.

Ninguna de estas decisiones queda cerrada únicamente con los resultados del notebook. El experimento sirve para comprender los mecanismos y construir una primera frontera, pero la configuración final debe repetirse con el volumen objetivo, el hardware de producción, el número real de threads, la concurrencia esperada, los filtros de negocio y una muestra representativa de consultas. También deben incluirse las etapas posteriores del pipeline, porque una mejora en la búsqueda ANN puede dejar de ser relevante si la latencia dominante se encuentra en la hidratación de metadatos o en el reranking.

Lo que sí queda establecido es un método de decisión. Flat actúa como oráculo y define el ranking exacto contra el que se comparan las alternativas. El recall cuantifica cuánta fidelidad se pierde al aproximar. Las curvas recall-latencia muestran qué configuraciones intercambian calidad por velocidad de manera eficiente. La memoria, la construcción y el ciclo de vida completan el coste técnico. Finalmente, el negocio fija la restricción que convierte esas mediciones en una elección concreta.

La pregunta final no es qué índice es mejor en términos absolutos, sino cuál es la estructura más sencilla que cumple, de forma reproducible, los requisitos completos del marketplace.